# VQA 학습 노트북 (통합판) · LoRA 파인튜닝 → 검증 → test 추론

이 노트북 하나로 제출에 쓴 **학습 모델을 모두** 다시 만들 수 있습니다. **1-1번 셀의 `RECIPE`에 실행 이름을 넣고** 위에서부터 실행하면, 그 실행과 같은 설정으로 학습 → 검증 → test 추론까지 하고 `OUT_DIR/<실행 이름>/`에 저장합니다.
(Flash-Next 학습은 `02_train_flashnext_runpod.ipynb`, 학습 없는 zero-shot 모델은 `03_*` 노트북입니다.)

**지원 모델 계열** — 프로세서 종류로 자동 판별합니다. 모델별 처리는 `MODEL_NOTES.md`에 정리했습니다.

| 계열 | 모델 예 | 해상도 조절 | 모델별 처리 |
|---|---|---|---|
| Qwen3.5 / 3.6 / 3.8 · Qwen3-VL | `Qwen/Qwen3.5-27B`, `Qwen/Qwen3.8-27B`, `Qwen/Qwen3.6-35B-A3B` | 이미지 픽셀 범위 (`min/max_pixels`) | 선형 어텐션 커널(`flash-linear-attention`) 설치, 생각 모드 끔 |
| Gemma 4 | `google/gemma-4-31B-it` | 이미지당 소프트 토큰 수 | HF 라이선스 동의 + 토큰 |
| InternVL3.5 | `OpenGVLab/InternVL3_5-38B-HF` | 448px 타일 개수 | — |
| HyperCLOVA X SEED Think | `naver-hyperclovax/HyperCLOVAX-SEED-Think-32B` | 이미지 픽셀 범위 | transformers 개발 버전, 생각 모드 끔 + 빈 생각 블록 닫기 (7-1번 셀) |

**핵심 설계**
- **정답 글자만 학습**: 추론 때와 똑같은 프롬프트를 넣고, 마지막 위치(답이 나올 자리)의 로짓에만 교차 엔트로피를 겁니다. 학습과 추론의 입력이 완전히 같습니다.
- **채점**: 답 자리에서 a·b·c·d(대소문자·앞 공백 변형 포함) 토큰 확률을 모아 네 보기 확률로 정규화합니다. 네 글자 확률 합(`letter_mass`)이 1에 가까워야 정상입니다.
- **선지 셔플 증강**: epoch마다 보기 순서를 무작위로 섞고 정답 글자를 그에 맞춰 바꿉니다(자리 편향 제거).
- **고정 검증셋**: train에서 정답 글자별 250문항씩 뽑은 1,000문항(시드 42, md5 `df05e4bc…`). 모든 실행이 같은 문항으로 비교됩니다.
- **끊겨도 이어서**: `SAVE_EVERY` 스텝마다 어댑터·옵티마이저·진행 상태를 저장합니다. 같은 셀들을 다시 실행하면 이어서 학습합니다.

**켜고 끄는 옵션** (1번 셀, 레시피가 덮어씀)

| 옵션 | 뜻 |
|---|---|
| `TRAIN_PRESET` | 학습 해상도 (`native` ≈ 원본, `up2.0` ≈ 2배) — 모델 계열별 값은 `RES_PRESETS*` |
| `USE_4BIT` | `"auto"`: 20B 이상 모델은 4bit NF4(QLoRA, 비전 타워·투영층·lm_head는 양자화 안 함) / `False`: bf16 |
| `VISION_LORA` | 비전 블록·투영층의 선형층에도 LoRA (학습률 = `LR x VIS_LR_SCALE`) |
| `USE_OCR` | 이미지 속 글자(OCR 결과)를 프롬프트 앞에 넣음 (`00b_ocr_paddle.ipynb` 결과 필요) |
| `AUX_GEN_CSV` | 보기 없는 단답형 보조 학습 (`00_data_prep.ipynb`가 만든 파일) |
| `INIT_ADAPTER` · `PLAN_EPOCH_OFFSET` | 기존 어댑터에서 이어학습 · 문항 순서 선택 |
| `EXTRA_TRAIN_CSV` | 추가 학습 데이터 (dev 의사 라벨, 12-2번 셀이 만듦) |
| `TRAIN_ON_FULL` (1-1번 `FULL_DATA`) | 검증 1,000문항까지 합쳐 6,714문항 학습 (제출용. 검증 점수는 의미 없음) |
| `TRAIN_SEED` · `IMG_AUG` · `TRAIN_PRESETS` | 학습 시드 · 밝기/대비 증강 · 다해상도 학습 |

**실행 순서**: ① `DRY_RUN = True`(기본)로 위에서부터 끝까지 실행 → 흐름·메모리·예상 시간 확인 (결과는 `<실행 이름>_dry` 폴더) → ② `DRY_RUN = False`로 바꿔 1번 셀부터 다시 실행 (본 학습).
HyperCLOVA X는 transformers 개발 버전을 설치하므로 **새 세션**에서 실행하세요.

**결과 파일** (`OUT_DIR/<실행 이름>/`, 조합 노트북이 이 이름으로 찾습니다)
- `adapter_ep<N>/` (LoRA 어댑터) · `train_log.json` (학습 설정·epoch별 검증 정확도)
- `preds_final_ep<N>__<해상도>__1000.csv` (검증 예측) · `preds_final_ep<N>_test__<해상도>__6714.csv` (test 예측)
- 열: `id, pred, p_a, p_b, p_c, p_d, letter_mass, img_tokens` · 단독 제출 파일 `submission_<실행>_final_ep<N>_<해상도>.csv`

## 1. 기본 설정
1-1번 셀에서 `RECIPE`를 고르면 그 실행의 값으로 덮어씁니다. `RECIPE = None`이면 여기 값 그대로 새 실험을 합니다.

In [ ]:
RUN_NAME   = "my_run"          # 실험 이름 = 저장 폴더 이름 (RECIPE를 고르면 레시피 이름으로 바뀜)
MODEL_ID   = "Qwen/Qwen3.5-27B"
DRY_RUN    = True              # True: 3스텝만 학습해 전체 흐름 점검 (본 학습 전 필수) / False: 본 학습

# ── 학습 ──
TRAIN_PRESET = "native"        # 학습 해상도 (아래 RES_PRESETS의 이름)
EPOCHS     = 1
MAX_STEPS  = None              # 숫자를 넣으면 그 스텝에서 학습을 끝냄. None이면 EPOCHS만큼
LR         = 1e-4
MICRO_BS   = 2                 # GPU에 한 번에 올리는 문항 수. 학습 중 OOM이면 자동으로 절반 + GRAD_ACCUM 2배 (유효 배치 유지)
GRAD_ACCUM = 8                 # 유효 배치 = MICRO_BS x GRAD_ACCUM = 16
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
WARMUP_RATIO = 0.05            # cosine 스케줄의 warmup 비율
SHUFFLE_OPTIONS = True         # 선지 셔플 증강
TRAIN_ON_FULL = False          # True: 검증셋까지 합쳐 6,714문항으로 학습 (1-1번 셀의 FULL_DATA가 켬)
VISION_LORA  = False           # True: 비전 블록·merger(투영층)의 선형층에도 LoRA (언어 LoRA는 그대로)
VIS_LR_SCALE = 0.2             # 비전 쪽 LoRA 학습률 = LR x 이 값
INIT_ADAPTER = None            # 이 어댑터 폴더에서 이어학습 (옵티마이저·스케줄러는 새로). 예: f"{OUT_DIR}/q35_27b_r16_native/adapter_ep1"
PLAN_EPOCH_OFFSET = 0          # 1이면 원 학습의 '2번째 epoch' 문항·보기 순서로 학습 (이어학습용)
EPOCH_END_EVAL = True          # False: epoch 끝의 학습 해상도 검증 평가 생략 (어댑터 저장은 그대로)
LOG_EVERY, EVAL_EVERY, SAVE_EVERY = 10, 100, 50    # 옵티마이저 스텝 단위
QUICK_EVAL_N = 300             # 학습 도중 빠른 점검용 검증 문항 수 (epoch 끝에는 1,000개 전체 평가)
TRAIN_SEED = 42                # 학습 순서·선지 셔플·LoRA 초기화 시드 (같은 레시피로 두 번째 모델을 만들 때 43 등)
IMG_AUG = False                # True: 학습 이미지에만 밝기·대비·채도·감마·JPEG 화질을 흔듦 (글자·위치는 그대로)
TRAIN_PRESETS = None           # 예: ["native", "up1.5", "up2.0"] → 마이크로 배치마다 학습 해상도를 무작위로 (다해상도 학습)
EXTRA_TRAIN_CSV = None         # 추가 학습 데이터 CSV (열: id,path,question,a,b,c,d,answer). 검증 문항은 자동 차단

# ── 프롬프트: OCR 글자 (00b_ocr_paddle.ipynb 결과) ──
USE_OCR       = False          # True: 이미지 속 글자 줄을 질문 앞에 넣음 (학습·검증·test 모두 같은 형식)
OCR_TAG       = "pvl16_spotting"   # OUT_DIR/ocr_paddle/<OCR_TAG>/ocr_text.csv
OCR_MAX_CHARS = 1000           # 넣을 글자 수 상한. 넘으면 질문·보기와 글자가 많이 겹치는 줄부터 남김
OCR_POS_TAGS  = True           # 줄마다 대략의 위치 "(위 왼쪽)"

# ── 보조 학습: 보기 없는 단답형 (00_data_prep.ipynb가 만든 파일) ──
AUX_GEN_CSV = None             # 열: id,path,question,answer_text. None이면 끔
AUX_GEN_EVERY = 5              # 객관식 마이크로 배치 N개마다 보조 마이크로 배치 1개 (4면 문항 비중 80:20)
AUX_GEN_WEIGHT = 1.0           # 보조 손실(정답 문자열 토큰 평균 교차 엔트로피) 가중치
AUX_MAX_ANS_TOKENS = 24        # 이보다 긴 정답 문자열은 보조 학습에서 제외
GEN_PROMPT_VERSION = "v1_en_sys"   # 보조 학습 지시문: "v1_en_sys"(영어) | "v2_ko_sys"(한국어)

# ── 평가 / 제출 ──
EVAL_PRESETS = ["up2.0"]       # 학습이 끝난 뒤 이 해상도들로 검증 1,000문항 평가
EVAL_TTA     = False           # 가장 좋았던 해상도에서 선지 순열 4종 평균(TTA)의 효과도 측정
MAKE_SUBMISSION = True         # 학습·검증 뒤 이어서 test 추론 → 제출 파일
TEST_PRESET  = "up2.0"         # test 추론 해상도 ("auto": 검증이 가장 좋았던 해상도)
TEST_TTA     = False
ZEROSHOT_CHECK_N = 0           # >0이면 학습 전(LoRA 0)에 검증 앞 N문항을 채점해 기본 모델 실력 확인

# ── 경로 · 하드웨어 ──
DRIVE_ZIP    = "/content/drive/MyDrive/ssafy-16-2-ai.zip"
ZIP_PASSWORD = "YOUR_ZIP_PASSWORD"
DATA_DIR     = "/content"
OUT_DIR      = "/content/drive/MyDrive/vqa_runs"
USE_4BIT   = "auto"            # "auto": VRAM 30GB 미만·bf16 미지원·20B 이상 모델이면 4bit NF4(QLoRA), 그 외 bf16 LoRA / False: bf16
BATCH_SIZE = "auto"            # 평가용 배치
SEED, VAL_SIZE = 42, 1000      # 검증 분할용 — 공통값이므로 바꾸지 마세요
TRAIN_LIMIT = None             # 디버그용: 학습 샘플 수 제한
SKIP_MISSING_IMAGES = False
HF_TOKEN   = ""                # Gemma처럼 라이선스 승인이 필요한 모델일 때 (비워 두면 Colab 보안 비밀 'HF_TOKEN')

RES_PRESETS = {                # Qwen·HCX 계열: 이미지 픽셀 수 범위 (min_pixels, max_pixels)
    "baseline384": (384 * 384, 384 * 384),
    "mid":         (256 * 28 * 28, 512 * 28 * 28),
    "native":      (256 * 28 * 28, 1280 * 28 * 28),
    "up1.5":       (1_550_000, 2_100_000),
    "up2.0":       (2_700_000, 3_400_000),
}
# Gemma 4: 이미지당 '소프트 토큰' 수 (280 ≈ 원본, 1120 ≈ 1.9배)
RES_PRESETS_GEMMA4 = {"baseline384": 70, "mid": 140, "native": 280, "up1.5": 560, "up2.0": 1120}
# InternVL: 448px 타일 개수 (타일 1개 = 이미지 토큰 256개 + 전체 축소본 1타일). 720x960 사진 기준 up2.0(12타일) ≈ 1.9배
RES_PRESETS_INTERNVL = {"baseline384": 1, "mid": 4, "native": 6, "up1.5": 9, "up2.0": 12, "up2.5": 16}
RES_PRESET = TRAIN_PRESET
import os; os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # GPU 메모리 단편화 완화 (torch import 전에 설정)
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")   # 허깅페이스 고속 다운로드

## 1-1. 레시피 — 제출에 쓴 실행을 이름으로 고르기
- 아래 `RECIPES`의 값은 각 실행의 `train_log.json`(학습 기록)에서 그대로 옮긴 것입니다. 정밀도(`USE_4BIT`)는 기록에 없어 실행 이름·학습 노트북 기준으로 적었습니다.
- `SYNC_FROM_LOG = True`면 `OUT_DIR/<RECIPE>/train_log.json`이 있을 때 **기록 값을 다시 한 번 대조**해서 다른 값이 있으면 기록 쪽으로 맞추고 출력합니다.
- 이미 끝난 실행 폴더(제출 파일이 있음)에는 덮어쓰지 않고 멈춥니다. 같은 설정으로 다시 만들어 비교하려면 `RUN_SUFFIX = "_repro"`처럼 붙이세요.
- 학습이 중간에 끊긴 폴더(체크포인트만 있음)는 이어서 학습합니다.
- `REUSE_FINISHED = True`: 끝난 실행을 **학습 없이** 불러와 11번 이후(평가·test·dev 의사 라벨)만 다시 돌립니다. 저장된 예측 파일은 그대로 쓰고(덮어쓰지 않음), `REPRO_CHECK_N`문항은 저장된 어댑터로 새로 채점해 저장된 예측과 같은지 확인합니다.

In [ ]:
RECIPE        = "q35_27b_r16_native_bf16"   # 아래 RECIPES의 이름. None이면 1번 셀 설정 그대로 (새 실험)
FULL_DATA     = False      # True: 같은 레시피를 검증 1,000문항까지 합친 6,714문항으로 학습 → 실행 이름 끝에 "_full"
SYNC_FROM_LOG = True       # True: OUT_DIR/<RECIPE>/train_log.json이 있으면 기록 값과 대조해 맞춤
RUN_SUFFIX    = ""         # 예: "_repro" → 원래 실행과 다른 폴더에 다시 만들기
REUSE_FINISHED = False     # True: 끝난 실행 폴더의 어댑터를 불러와 학습 없이 평가·test·dev 의사 라벨만 (예측 파일은 이어쓰기 규칙대로 재사용)
REPRO_CHECK_N  = 100       # REUSE_FINISHED일 때 검증 앞 N문항을 저장된 어댑터로 새로 채점해 저장된 예측과 대조 (0이면 생략)

RECIPES = {
    'q35_27b_r16_native':           dict(MODEL_ID='Qwen/Qwen3.5-27B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT='auto'),
    'q35_27b_r16_native_bf16':      dict(MODEL_ID='Qwen/Qwen3.5-27B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT=False),
    'q35_27b_r16_native_ocr_bf16':  dict(MODEL_ID='Qwen/Qwen3.5-27B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT=False, USE_OCR=True, OCR_TAG='pvl16_spotting', OCR_MAX_CHARS=1000, OCR_POS_TAGS=True),
    'q35_27b_r16_native_full':      dict(MODEL_ID='Qwen/Qwen3.5-27B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT=False, TRAIN_ON_FULL=True),
    'q35_27b_r16_review1_up20':     dict(MODEL_ID='Qwen/Qwen3.5-27B', TRAIN_PRESET='up2.0', MICRO_BS=1, GRAD_ACCUM=16, USE_4BIT='auto'),
    'q35_27b_r16_review1_up20_vis': dict(MODEL_ID='Qwen/Qwen3.5-27B', TRAIN_PRESET='up2.0', MICRO_BS=1, GRAD_ACCUM=16, USE_4BIT='auto', VISION_LORA=True),
    'q35_27b_r16_e4ctrl_native':    dict(MODEL_ID='Qwen/Qwen3.5-27B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT='auto', MAX_STEPS=178, LR=2e-05, INIT_ADAPTER=f"{OUT_DIR}/q35_27b_r16_native/adapter_ep1", PLAN_EPOCH_OFFSET=1, GEN_PROMPT_VERSION='v2_ko_sys', EPOCH_END_EVAL=False, EVAL_EVERY=10**9),
    'q35_27b_r16_e4aux20_native':   dict(MODEL_ID='Qwen/Qwen3.5-27B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT='auto', MAX_STEPS=178, LR=2e-05, AUX_GEN_CSV=f"{OUT_DIR}/aux/aux_open_v1.csv", AUX_GEN_EVERY=4, INIT_ADAPTER=f"{OUT_DIR}/q35_27b_r16_native/adapter_ep1", PLAN_EPOCH_OFFSET=1, GEN_PROMPT_VERSION='v2_ko_sys', EPOCH_END_EVAL=False, EVAL_EVERY=10**9),
    'q35_9b_r16_native':            dict(MODEL_ID='Qwen/Qwen3.5-9B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT='auto', EPOCHS=2),
    'q36_35ba3b_r16_native_bf16':   dict(MODEL_ID='Qwen/Qwen3.6-35B-A3B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT=False),
    'q38_27b_r16_native':           dict(MODEL_ID='Qwen/Qwen3.8-27B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT='auto'),
    'q38_27b_r16_native_43':        dict(MODEL_ID='Qwen/Qwen3.8-27B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT='auto', TRAIN_SEED=43),
    'q38_27b_r16_native_bf16':      dict(MODEL_ID='Qwen/Qwen3.8-27B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT=False),
    'q38_27b_r16_native_dev':       dict(MODEL_ID='Qwen/Qwen3.8-27B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT='auto', EXTRA_TRAIN_CSV=f"{OUT_DIR}/dev_pseudo__gemma4_31b_r16_up20__up2.0__c0.8_v3_agree.csv"),
    'q38_27b_r16_native_full':      dict(MODEL_ID='Qwen/Qwen3.8-27B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT=False, TRAIN_ON_FULL=True),
    'q3vl_8b_r16_native':           dict(MODEL_ID='Qwen/Qwen3-VL-8B-Instruct', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT='auto', EPOCHS=2),
    'q3vl_32b_r16_native_full':     dict(MODEL_ID='Qwen/Qwen3-VL-32B-Instruct', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT=False, TRAIN_ON_FULL=True),
    'gemma4_31b_r16_up20':          dict(MODEL_ID='google/gemma-4-31B-it', TRAIN_PRESET='up2.0', MICRO_BS=1, GRAD_ACCUM=16, USE_4BIT='auto'),
    'gemma4_31b_r16_up20_bf16':     dict(MODEL_ID='google/gemma-4-31B-it', TRAIN_PRESET='up2.0', MICRO_BS=1, GRAD_ACCUM=16, USE_4BIT=False),
    'gemma4_31b_r16_up20_ocr_bf16': dict(MODEL_ID='google/gemma-4-31B-it', TRAIN_PRESET='up2.0', MICRO_BS=1, GRAD_ACCUM=16, USE_4BIT=False, USE_OCR=True, OCR_TAG='pvl16_spotting', OCR_MAX_CHARS=1000, OCR_POS_TAGS=True),
    'gemma4_31b_r16_up20_dev':      dict(MODEL_ID='google/gemma-4-31B-it', TRAIN_PRESET='up2.0', MICRO_BS=1, GRAD_ACCUM=16, USE_4BIT='auto', EXTRA_TRAIN_CSV=f"{OUT_DIR}/dev_pseudo__gemma4_31b_r16_up20__up2.0__c0.8_v3_agree.csv"),
    'gemma4_31b_r16_up20_full':     dict(MODEL_ID='google/gemma-4-31B-it', TRAIN_PRESET='up2.0', MICRO_BS=1, GRAD_ACCUM=16, USE_4BIT=False, TRAIN_ON_FULL=True),
    'gemma4_26ba4b_r16_up20_bf16':  dict(MODEL_ID='google/gemma-4-26B-A4B-it', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT=False),
    'hcx32b_r16_native':            dict(MODEL_ID='naver-hyperclovax/HyperCLOVAX-SEED-Think-32B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT='auto'),
    'hcx32b_r16_native_bf16':       dict(MODEL_ID='naver-hyperclovax/HyperCLOVAX-SEED-Think-32B', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT=False),
    'ivl35_38b_r16_native':         dict(MODEL_ID='OpenGVLab/InternVL3_5-38B-HF', TRAIN_PRESET='native', MICRO_BS=2, GRAD_ACCUM=8, USE_4BIT='auto'),
}
RECIPE_NOTES = {
    'q35_27b_r16_native':           'Qwen3.5-27B · native 학습 · 4bit NF4 (기본 레시피)',
    'q35_27b_r16_native_bf16':      '위와 같고 정밀도만 bf16',
    'q35_27b_r16_native_ocr_bf16':  '위 bf16 + 프롬프트에 OCR 글자',
    'q35_27b_r16_native_full':      '검증 1,000문항까지 합친 6,714문항 학습 (제출용, bf16)',
    'q35_27b_r16_review1_up20':     '학습 해상도 up2.0',
    'q35_27b_r16_review1_up20_vis': '학습 해상도 up2.0 + 비전 LoRA (비전 학습률 = LR x 0.2)',
    'q35_27b_r16_e4ctrl_native':    'q35_27b_r16_native 어댑터에서 0.5 epoch 이어학습 (LR 2e-5, 2번째 epoch 순서)',
    'q35_27b_r16_e4aux20_native':   '위 이어학습 + 보기 없는 단답형 보조학습 20% (한국어 지시문)',
    'q35_9b_r16_native':            'Qwen3.5-9B · 2 epoch (검증으로 epoch 선택)',
    'q36_35ba3b_r16_native_bf16':   'Qwen3.6-35B-A3B (MoE) · bf16',
    'q38_27b_r16_native':           'Qwen3.8-27B · native 학습 · 4bit NF4',
    'q38_27b_r16_native_43':        '위와 같고 학습 시드만 43',
    'q38_27b_r16_native_bf16':      '위와 같고 정밀도만 bf16',
    'q38_27b_r16_native_dev':       '+ dev 의사 라벨 추가 학습',
    'q38_27b_r16_native_full':      '검증 포함 6,714문항 학습 (제출용, bf16)',
    'q3vl_8b_r16_native':           'Qwen3-VL-8B · 2 epoch (검증으로 epoch 선택)',
    'q3vl_32b_r16_native_full':     'Qwen3-VL-32B · 검증 포함 6,714문항 학습 (bf16)',
    'gemma4_31b_r16_up20':          'Gemma 4 31B · 학습 해상도 up2.0 · 4bit NF4',
    'gemma4_31b_r16_up20_bf16':     '위와 같고 정밀도만 bf16',
    'gemma4_31b_r16_up20_ocr_bf16': '위 bf16 + 프롬프트에 OCR 글자',
    'gemma4_31b_r16_up20_dev':      '+ dev 의사 라벨 추가 학습',
    'gemma4_31b_r16_up20_full':     '검증 포함 6,714문항 학습 (제출용, bf16)',
    'gemma4_26ba4b_r16_up20_bf16':  'Gemma 4 26B-A4B (MoE) · native 학습 · bf16',
    'hcx32b_r16_native':            'HyperCLOVA X SEED Think 32B · native · 4bit NF4',
    'hcx32b_r16_native_bf16':       '위와 같고 정밀도만 bf16',
    'ivl35_38b_r16_native':         'InternVL3.5-38B · native(6타일) · 4bit NF4',
}

import os, json
LOG_SYNCED = {}
if RECIPE:
    assert RECIPE in RECIPES, f"RECIPES에 없는 이름입니다: {RECIPE}\n목록: {list(RECIPES)}"
    for _k, _v in RECIPES[RECIPE].items(): globals()[_k] = _v
    RUN_NAME = RECIPE
    _lp = os.path.join(OUT_DIR, RECIPE, "train_log.json")
    _log = json.load(open(_lp, encoding="utf-8")) if os.path.exists(_lp) else {}
    if SYNC_FROM_LOG and _log:
        _cfg = dict(_log.get("config") or {})
        _SYNC = ["MODEL_ID", "TRAIN_PRESET", "EPOCHS", "MAX_STEPS", "LR", "LORA_R", "LORA_ALPHA", "SHUFFLE_OPTIONS", "TRAIN_SEED", "TRAIN_ON_FULL",
                 "EXTRA_TRAIN_CSV", "IMG_AUG", "TRAIN_PRESETS", "AUX_GEN_CSV", "AUX_GEN_EVERY", "AUX_GEN_WEIGHT", "VISION_LORA", "VIS_LR_SCALE",
                 "INIT_ADAPTER", "PLAN_EPOCH_OFFSET", "GEN_PROMPT_VERSION", "OCR_TAG", "OCR_MAX_CHARS", "OCR_POS_TAGS"]
        for _k in _SYNC:
            if _k in _cfg and _cfg[_k] != globals().get(_k):
                if _k in ("OCR_TAG", "OCR_MAX_CHARS", "OCR_POS_TAGS") and not str(_cfg.get("PROMPT_VERSION", "")).startswith("ocr_"): continue
                LOG_SYNCED[_k] = (globals().get(_k), _cfg[_k]); globals()[_k] = _cfg[_k]
        if "PROMPT_VERSION" in _cfg and str(_cfg["PROMPT_VERSION"]).startswith("ocr_") != bool(USE_OCR):
            LOG_SYNCED["USE_OCR"] = (USE_OCR, not USE_OCR); USE_OCR = not USE_OCR
        if "STEP_SAMPLES" in _cfg and int(_cfg["STEP_SAMPLES"]) != MICRO_BS * GRAD_ACCUM:
            _ss = int(_cfg["STEP_SAMPLES"]); assert _ss % MICRO_BS == 0, f"유효 배치 {_ss}를 MICRO_BS={MICRO_BS}로 나눌 수 없습니다"
            LOG_SYNCED["GRAD_ACCUM"] = (GRAD_ACCUM, _ss // MICRO_BS); GRAD_ACCUM = _ss // MICRO_BS
        assert int(_cfg.get("SEED", 42)) == 42, f"검증 분할 시드가 공통값(42)이 아닙니다: {_cfg.get('SEED')}"
        _unsup = {k: _cfg[k] for k in ("LOSS_MODE", "SAVE_FRACS") if _cfg.get(k) not in (None, "full")}
        if _unsup: print("⚠️ 이 노트북에 없는 학습 설정이 기록에 있습니다 (결과가 다를 수 있음):", _unsup)
        print(f"학습 기록 대조: {_lp} → " + (f"다른 값 {len(LOG_SYNCED)}개를 기록대로 맞춤" if LOG_SYNCED else "레시피와 모두 같음 ✅"))
        for _k, (_a, _b) in LOG_SYNCED.items(): print(f"  {_k:<18} {_a!r} → {_b!r}")
    elif SYNC_FROM_LOG:
        print(f"ℹ️ 학습 기록이 없습니다 ({_lp}) → 레시피 값 그대로 씁니다")
    if FULL_DATA:
        assert not TRAIN_ON_FULL, f"{RECIPE}는 이미 전체 데이터 학습 레시피입니다"
        _ea = {int(k): v for k, v in (_log.get("epoch_acc") or {}).items()}
        _best = max(_ea, key=lambda e: (_ea[e], e)) if _ea else None
        if _best is not None and _best != int(EPOCHS):
            raise RuntimeError(f"{RECIPE}는 EPOCHS={EPOCHS}로 학습하고 검증으로 epoch {_best}를 골랐습니다. "
                               "전체 데이터에서는 검증으로 epoch를 고를 수 없어 멈춥니다 → EPOCHS를 그 값으로 바꾸고 다시 실행하세요")
        if MAX_STEPS:
            _old = int(MAX_STEPS); MAX_STEPS = int(round(_old * 6714 / 5714))
            print(f"⚠️ 원래 실행은 MAX_STEPS={_old}에서 끊었습니다 → 데이터가 늘어난 비율(6,714/5,714)만큼 {MAX_STEPS}로 늘립니다")
        TRAIN_ON_FULL = True; RUN_NAME = RECIPE + "_full"
        EVAL_PRESETS, EVAL_TTA, MAKE_SUBMISSION, TEST_PRESET, TEST_TTA = ["up2.0"], False, True, "up2.0", False
    RUN_NAME += RUN_SUFFIX
    RES_PRESET = TRAIN_PRESET
    print(f"\n레시피 {RECIPE}: {RECIPE_NOTES.get(RECIPE, '')}")
else:
    print("RECIPE = None → 1번 셀 설정 그대로 (새 실험). 레시피 목록:")
    for _n, _t in RECIPE_NOTES.items(): print(f"  {_n:<30} {_t}")
_prev = os.path.join(OUT_DIR, RUN_NAME)
_done = os.path.isdir(_prev) and os.path.exists(os.path.join(_prev, "train_log.json")) and any(f.startswith("adapter_ep") for f in os.listdir(_prev))
if REUSE_FINISHED:
    assert not DRY_RUN and _done, f"REUSE_FINISHED: 끝난 실행 폴더(train_log.json + adapter_ep*)가 없습니다: {_prev} (DRY_RUN = False여야 함)"
    print(f"↩️ 끝난 실행을 학습 없이 다시 씀: {_prev}")
elif not DRY_RUN and os.path.isdir(_prev) and any(f.startswith("submission_") and f.endswith(".csv") for f in os.listdir(_prev)):
    raise RuntimeError(f"{_prev}는 이미 끝난 실행입니다 (제출 파일 있음). 덮어쓰지 않도록 멈춥니다 → 다시 만들어 비교하려면 RUN_SUFFIX = \"_repro\", "
                       "저장된 어댑터로 평가·test·dev 의사 라벨만 다시 돌리려면 REUSE_FINISHED = True")
_SHOW = ["RUN_NAME", "MODEL_ID", "DRY_RUN", "TRAIN_PRESET", "EPOCHS", "MAX_STEPS", "LR", "MICRO_BS", "GRAD_ACCUM", "USE_4BIT", "TRAIN_SEED", "TRAIN_ON_FULL",
         "VISION_LORA", "USE_OCR", "AUX_GEN_CSV", "AUX_GEN_EVERY", "INIT_ADAPTER", "PLAN_EPOCH_OFFSET", "EPOCH_END_EVAL", "GEN_PROMPT_VERSION",
         "EXTRA_TRAIN_CSV", "EVAL_PRESETS", "MAKE_SUBMISSION", "TEST_PRESET"]
print("최종 설정:"); [print(f"  {_k:<18} = {globals()[_k]!r}") for _k in _SHOW]

## 2. 라이브러리 설치 (버전 고정)
HyperCLOVA X만 transformers 개발 버전(검증한 커밋)을 쓰고, 나머지는 transformers 5.17.0입니다. 모델 계열을 바꿀 때는 **세션을 다시 시작**하고 처음부터 실행하세요.

In [ ]:
import re
_HCX = "hyperclova" in str(MODEL_ID).lower()
# HyperCLOVA X 비전 모델은 transformers 정식 5.17에 없고 개발 버전(main)에만 있음 → 검증한 커밋으로 고정
TF_PIN = ("transformers @ git+https://github.com/huggingface/transformers.git@98d39824ed30e684e5122d04a2d9564efffc4965" if _HCX
          else "transformers==5.17.0")
print("transformers:", TF_PIN)
!pip -q install "{TF_PIN}" "accelerate==1.15.0" "bitsandbytes==0.50.2" "peft==0.21.0"
# Colab에 기본 설치된 구버전 torchao가 있으면 peft 0.21이 LoRA를 붙이는 순간 ImportError → 쓰지 않으므로 제거
!pip -q uninstall -y torchao
# Qwen3.5 이후의 선형 어텐션용 Triton 커널. 없으면 순수 PyTorch 대체 구현으로 돌아 OOM·저속
if re.search(r"Qwen3\.[5-9]", MODEL_ID):      # Qwen3.5 / 3.6 / 3.8 — 같은 선형 어텐션 구조(model_type qwen3_5, qwen3_5_moe)
    !pip -q install "flash-linear-attention==0.5.2"

## 3. 데이터 준비 (Drive의 대회 zip → `DATA_DIR`)

In [ ]:
import os, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
    print("압축 해제 중... (몇 분 걸립니다)")
    r = subprocess.run(["7z", "x", "-y", f"-p{ZIP_PASSWORD}", f"-o{DATA_DIR}", DRIVE_ZIP],
                       capture_output=True, text=True)
    print(r.stdout[-400:]); assert r.returncode == 0, r.stderr[-800:]
if not HF_TOKEN and IN_COLAB:
    try:
        from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception: pass
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN, add_to_git_credential=False); print("Hugging Face 로그인 완료")
os.makedirs(OUT_DIR, exist_ok=True)
for f in ["train.csv", "test.csv", "sample_submission.csv"]:
    assert os.path.exists(os.path.join(DATA_DIR, f)), f"{f} 가 {DATA_DIR} 에 없습니다. DATA_DIR / 압축 비밀번호를 확인하세요."
print("데이터 OK:", DATA_DIR, "| 결과 저장:", OUT_DIR)

## 4. 공통 import · CSV 로드

In [ ]:
import re, json, time, math, hashlib, random, datetime, platform
import numpy as np, pandas as pd, torch
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

def _fix_nvrtc():
    # 일부 Colab 이미지(새 CUDA)에서 GPU 정수 텐서의 prod가 런타임 컴파일(NVRTC) 도중 내장 라이브러리를 못 찾아 실패함.
    # 1) 그 라이브러리를 경로로 직접 불러오고, 2) 그래도 안 되면 정수 prod만 CPU에서 계산하도록 우회 (결과는 완전히 같음)
    if not torch.cuda.is_available(): return
    import glob, ctypes
    def _ok():
        try: torch.tensor([[1, 2, 3]], device="cuda").prod(-1).item(); return True, ""
        except RuntimeError as e: return False, str(e)
    good, msg = _ok()
    if good: return
    need = re.findall(r"libnvrtc-builtins\.so[\d.]*", msg)
    pats = ["/usr/local/lib/python3*/dist-packages/nvidia/**/libnvrtc-builtins.so*", "/usr/local/lib/python3*/site-packages/nvidia/**/libnvrtc-builtins.so*",
            "/usr/local/cuda*/lib64/libnvrtc-builtins.so*", "/usr/lib/x86_64-linux-gnu/libnvrtc-builtins.so*"]
    found = sorted({f for p in pats for f in glob.glob(p, recursive=True)})
    for f in found:
        if not need or any(os.path.basename(f).startswith(n) for n in need):
            try: ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
            except OSError: pass
    if _ok()[0]: print("✅ NVRTC 내장 라이브러리를 직접 불러와 해결:", need); return
    _orig_prod = torch.Tensor.prod
    def _prod(self, *a, **k):
        if self.is_cuda and not (self.is_floating_point() or self.is_complex()):
            return _orig_prod(self.cpu(), *a, **k).to(self.device)
        return _orig_prod(self, *a, **k)
    torch.Tensor.prod = _prod
    print("⚠️ GPU 정수 prod를 CPU에서 계산하도록 우회 (결과 동일) | 확인:", _ok()[0], "| 찾은 라이브러리:", found or "없음")
_fix_nvrtc()

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sub_df   = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
dev_df   = pd.read_csv(os.path.join(DATA_DIR, "dev.csv")) if os.path.exists(os.path.join(DATA_DIR, "dev.csv")) else None
for c in ["question", "a", "b", "c", "d"]:
    train_df[c] = train_df[c].astype(str); test_df[c] = test_df[c].astype(str)
train_df["answer"] = train_df["answer"].astype(str).str.strip().str.lower()
assert set(train_df.answer.unique()) <= set("abcd"), train_df.answer.unique()
assert list(test_df.id) == list(sub_df.id), "test.csv 와 sample_submission.csv 의 id 순서가 다릅니다"
print(f"train {len(train_df):,} | test {len(test_df):,} | dev {0 if dev_df is None else len(dev_df):,}")
print("정답 분포:", train_df.answer.value_counts().sort_index().to_dict())

def img_path(rel):            # CSV의 path 컬럼은 DATA_DIR 기준 상대경로
    return os.path.join(DATA_DIR, rel)

## 5. 고정 검증셋 분할 (모든 실행 공통)

In [ ]:
EXPECTED_SPLIT_MD5 = "df05e4bc19d27ebceeef2570672149c2"

def make_split(df, val_size=VAL_SIZE, seed=SEED):
    rng = np.random.RandomState(seed)
    val_idx = []
    for letter in "abcd":
        idx = np.where(df.answer.values == letter)[0]
        rng.shuffle(idx)
        val_idx.extend(idx[: val_size // 4].tolist())
    val_idx = sorted(val_idx)
    mask = np.zeros(len(df), dtype=bool); mask[val_idx] = True
    return df[~mask].reset_index(drop=True), df[mask].reset_index(drop=True)

tr_df, val_df = make_split(train_df)
split_md5 = hashlib.md5(",".join(val_df.id.tolist()).encode()).hexdigest()
print(f"train {len(tr_df):,} / valid {len(val_df):,} | split md5 = {split_md5}")
print("valid 정답 분포:", val_df.answer.value_counts().sort_index().to_dict())
if split_md5 == EXPECTED_SPLIT_MD5:
    print("✅ 팀 공통 분할과 일치")
else:
    print("⚠️⚠️ 검증 분할이 기준과 다릅니다 → 팀원 간 점수 비교 불가. train.csv 원본/SEED/VAL_SIZE를 확인하세요.")
val_df[["id"]].to_csv(os.path.join(OUT_DIR, "valid_ids.csv"), index=False)

## 6. 프롬프트 (+ OCR 글자 선택)
- system: 한 글자로만 답하라는 지시 (영어 한 줄). user: 이미지 → 질문 → `(a)~(d)` 보기 → "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
- `USE_OCR = True`면 질문 앞에 `[이미지 속 글자 — OCR 자동 인식 결과 …]` 블록이 붙습니다. 보기 순서와 무관한 블록이라 선지를 섞어도 같습니다.
- system 역할을 못 푸는 템플릿은 7-1번 셀이 자동으로 판별해 지시문을 문자열이나 user 메시지 앞에 넣습니다.

In [ ]:
PROMPT_VERSION = f"ocr_v2_{OCR_TAG}" if USE_OCR else "baseline_v1"      # 학습 기록에 남는 프롬프트 이름
SYSTEM_INSTRUCT = ("You are a helpful visual question answering assistant. "
                   "Answer using exactly one letter among a, b, c, or d. No explanation.")
LETTERS = "abcd"

# ── OCR 결과 불러오기 (USE_OCR일 때만): 00b_ocr_paddle.ipynb가 만든 OUT_DIR/ocr_paddle/<OCR_TAG>/ocr_text.csv ──
def _clean_line(x):                                          # OCR이 생성 루프에 빠진 흔적 정리
    y = re.sub(r"(.)\1{5,}", r"\1" * 6, x)                   # 같은 글자가 6번 넘게 이어지면 6번으로
    y = re.sub(r"(.{2,12}?)\1{3,}", r"\1\1\1", y)            # 짧은 조각이 4번 넘게 반복되면 3번으로
    return y.strip()
def _clean_lines(lines):
    out, seen = [], {}
    for t, x in lines:
        y = _clean_line(x)
        if not y or len(y) < 0.4 * len(x): continue            # 대부분이 반복이던 줄은 버림
        seen[y] = seen.get(y, 0) + 1
        if seen[y] > 3: continue                                # 같은 글자 줄은 3번까지
        out.append((t, y))
    return out
def _norm(s): return re.sub(r"[\s\W_]+", "", str(s).lower())
def _bg(s): return {s[i:i + 2] for i in range(len(s) - 1)} if len(s) > 1 else ({s} if s else set())
OCR_HEADER = "[이미지 속 글자 — OCR 자동 인식 결과. 틀리거나 빠진 글자가 있을 수 있음]"
OCR_LINES, OCR_INFO = {}, {}
if USE_OCR:
    OCR_FILE = os.path.join(OUT_DIR, "ocr_paddle", OCR_TAG, "ocr_text.csv")
    assert os.path.exists(OCR_FILE), f"OCR 결과가 없습니다: {OCR_FILE} → 00b_ocr_paddle.ipynb를 먼저 끝까지 실행하세요"
    _ocr = pd.read_csv(OCR_FILE, keep_default_na=False)
    _raw_lines = {p: [(t or None, str(x)) for t, x in json.loads(j)] for p, j in zip(_ocr.path, _ocr.lines_json)}
    OCR_LINES = {p: _clean_lines(v) for p, v in _raw_lines.items()}
    _need = {"train": list(train_df.path)}
    if MAKE_SUBMISSION: _need["test"] = list(test_df.path)
    if EXTRA_TRAIN_CSV: _need["추가 학습 데이터"] = list(pd.read_csv(EXTRA_TRAIN_CSV).path)
    for _k, _ps in _need.items():
        _miss = [p for p in _ps if p not in OCR_LINES]
        assert not _miss, f"OCR 결과가 없습니다: {_k} 이미지 {len(_miss)}장 (예: {_miss[:3]}) → 00b_ocr_paddle.ipynb의 SPLITS·샤드 완료 여부를 확인하세요"
    _info_fp = os.path.join(os.path.dirname(OCR_FILE), "ocr_info.json")
    OCR_INFO = json.load(open(_info_fp)) if os.path.exists(_info_fp) else {}
    _nc = pd.to_numeric(_ocr.n_chars, errors="coerce").fillna(0)
    print(f"OCR {OCR_TAG}: {len(OCR_LINES):,}장 | {OCR_INFO.get('OCR_MODEL_ID', '?')} · {OCR_INFO.get('OCR_TASK', '?')} | "
          f"이미지당 글자 수 중앙값 {_nc.median():.0f} · {OCR_MAX_CHARS}자 초과 {(_nc > OCR_MAX_CHARS).mean() * 100:.1f}%")

def ocr_block(row):
    # 선지 순서와 무관 (학습 중 선지를 섞어도 같은 블록)
    lines = OCR_LINES.get(row.get("path") if hasattr(row, "get") else None, [])
    fmt = [((f"({t}) " if (OCR_POS_TAGS and t) else "") + x)[:200] for t, x in lines]
    if sum(len(f) + 1 for f in fmt) > OCR_MAX_CHARS:            # 길면 질문·보기와 겹치는 두 글자 조각이 많은 줄부터 남기고 원래 순서로
        q = _bg(_norm(" ".join([str(row["question"])] + [str(row[k]) for k in LETTERS])))
        score = [len(_bg(_norm(x)) & q) for _, x in lines]
        keep, used = set(), 0
        for i in sorted(range(len(fmt)), key=lambda i: (-score[i], i)):
            if used + len(fmt[i]) + 1 <= OCR_MAX_CHARS: keep.add(i); used += len(fmt[i]) + 1
        fmt = [fmt[i] for i in sorted(keep)] + ["…(일부 줄 생략)"]
    return OCR_HEADER + "\n" + ("\n".join(fmt) if fmt else "(인식된 글자 없음)")

def build_mc_prompt(question, options):
    a, b, c, d = options
    return (f"{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")

def build_messages(row, perm=(0, 1, 2, 3)):
    # perm[k] = 화면의 k번째 자리에 놓을 '원래 선지'의 인덱스
    opts = [str(row[LETTERS[j]]) for j in perm]
    user_text = build_mc_prompt(str(row["question"]), opts)
    if USE_OCR: user_text = ocr_block(row) + "\n\n" + user_text          # OCR 블록 → 질문·보기·답 지시
    if globals().get("USE_SYSTEM_ROLE", True):
        _sys = SYSTEM_INSTRUCT if globals().get("SYSTEM_AS_STRING", False) else [{"type": "text", "text": SYSTEM_INSTRUCT}]   # 목록형 system을 못 푸는 템플릿(EXAONE 등)은 문자열로
        return [{"role": "system", "content": _sys},
                {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]}]
    # system 역할이 없는 템플릿(일부 Gemma 등): 지시문을 user 메시지 앞에 붙임
    return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": SYSTEM_INSTRUCT + "\n\n" + user_text}]}]

def load_image(rel):
    im = Image.open(img_path(rel))
    im = ImageOps.exif_transpose(im)      # 촬영 방향(EXIF) 보정 — 없으면 그대로
    return im.convert("RGB")

## 7-1. 장치·정밀도 · 프로세서 · 모델별 프롬프트 보정
이 셀이 모델마다 다른 처리를 정합니다(자세한 내용은 `MODEL_NOTES.md`).
- **생각 모드 끄기**: 템플릿 변수 `enable_thinking=False`(Qwen·Gemma) / `thinking=False`(HyperCLOVA X).
- **system 형식 판별**: 목록형 system 내용이 풀리지 않고 글자 그대로 찍히는 템플릿이면 문자열로, 그래도 안 되면 user 메시지 앞에 붙입니다.
- **빈 생각 블록 닫기**: HyperCLOVA X 템플릿은 생각 블록을 연 채(`<think>\n`) 끝나므로 `\n</think>\n\n`을 붙여 첫 토큰이 곧 답 글자가 되게 합니다.
- 확인할 것: `프롬프트 미리보기(끝)` — Qwen은 `</think>\n\n`, HyperCLOVA X는 `<think>\n\n</think>\n\n`으로 끝나야 합니다. 첫 학습 로그의 loss가 1보다 한참 작아야 정상입니다(20 이상이면 답 자리가 틀린 것).

In [ ]:
import torch.nn.functional as F
from transformers import AutoProcessor, AutoModelForImageTextToText, AutoModelForCausalLM, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel, set_peft_model_state_dict, load_peft_weights, prepare_model_for_kbit_training
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)

import gc
for _v in ("model", "base", "optimizer", "scheduler", "params", "loss", "enc", "logits", "out", "gnorm"):   # 이전 모델·중단된 학습의 그래프가 GPU에 남아 OOM 나는 것 방지
    if _v in globals(): del globals()[_v]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_NAME = torch.cuda.get_device_name(0); cap = torch.cuda.get_device_capability(0)
    BF16_NATIVE = cap[0] >= 8
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
else:
    GPU_NAME, BF16_NATIVE, VRAM_GB = "cpu", False, 0.0
if DEVICE == "cpu":   DTYPE, use_4bit = torch.float32, False
elif BF16_NATIVE:
    _big = re.search(r"(?<![\d.])(2\d|3\d|7\d)B", str(MODEL_ID)) is not None
    DTYPE, use_4bit = torch.bfloat16, (USE_4BIT is True or (USE_4BIT == "auto" and (VRAM_GB < 30 or _big)))
else:                 DTYPE, use_4bit = torch.float16, (USE_4BIT is not False)
USE_FP16_SCALER = (DEVICE == "cuda" and not BF16_NATIVE)
BS = (8 if VRAM_GB >= 38 else 4 if VRAM_GB >= 20 else 2) if BATCH_SIZE == "auto" else int(BATCH_SIZE)
print(f"GPU: {GPU_NAME} ({VRAM_GB:.0f}GB) | dtype={DTYPE} | 4bit={use_4bit} | 평가 배치={BS} | 학습 배치={MICRO_BS}x{GRAD_ACCUM}")

PRECISION = "nf4" if use_4bit else str(DTYPE).replace("torch.", "")      # 학습 기록에 남김

import shutil
RUN_DIR = os.path.join(OUT_DIR, RUN_NAME + ("_dry" if DRY_RUN else ""))
CKPT_DIR = os.path.join(RUN_DIR, "ckpt_last")
if DRY_RUN and os.path.exists(RUN_DIR): shutil.rmtree(RUN_DIR)          # 점검용 실행은 항상 처음부터
os.makedirs(RUN_DIR, exist_ok=True)
if not os.path.exists(CKPT_DIR) and os.path.exists(os.path.join(CKPT_DIR + "_tmp", "trainer_state.pt")):
    os.replace(CKPT_DIR + "_tmp", CKPT_DIR)                              # 저장 직후 교체 단계에서 끊긴 경우 복구

load_kw = dict(dtype=DTYPE, attn_implementation="sdpa")
if DEVICE == "cuda": load_kw["device_map"] = {"": 0}
if use_4bit:
    load_kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE,
        llm_int8_skip_modules=["lm_head", ".*visual.*", ".*vision_tower.*", ".*vision_model.*", ".*multi_modal_projector.*", ".*projector.*", ".*embed_vision.*", ".*audio_tower.*", ".*embed_audio.*"])

_TRC = bool(globals().get("TRUST_REMOTE_CODE", False))       # 저장소의 커스텀 코드로 올라오는 모델(A.X 등)
processor = AutoProcessor.from_pretrained(MODEL_ID, **({"trust_remote_code": True} if _TRC else {}))
processor.tokenizer.padding_side = "left"
tok = processor.tokenizer
_pname = type(processor).__name__
FAMILY = "gemma4" if _pname.startswith("Gemma4") else "internvl" if _pname.startswith("InternVL") else "qwen"     # 프로세서 계열 (해상도 인자·이미지 전달 방식이 다름)
_ip = getattr(processor, "image_processor", None)
if _TRC and FAMILY == "qwen" and not (hasattr(_ip, "merge_size") or hasattr(_ip, "min_pixels")):
    FAMILY = "generic"      # 커스텀 프로세서: 해상도 인자를 모르므로 모델 기본 전처리를 그대로 씀 (up 프리셋은 이미지 자체를 확대)
# ═══════════ 모델별 처리 (MODEL_NOTES.md) ═══════════
# 1) 생각(추론) 모드 끄기: 템플릿 변수 이름이 enable_thinking(Qwen·Gemma) 또는 thinking(HyperCLOVA X)
# 2) system 내용 형식: 목록형 system을 못 풀고 글자 그대로 찍는 템플릿은 문자열로, 그래도 안 되면 user 메시지 앞에 붙임
# 3) 빈 생각 블록 닫기: 템플릿이 '<think>\n'으로 끝나면(HyperCLOVA X) '\n</think>\n\n'을 붙여 첫 토큰에서 바로 답하게 함
IMAGE_TOKEN_ID = getattr(processor, "image_token_id", None)
if IMAGE_TOKEN_ID is None: IMAGE_TOKEN_ID = tok.convert_tokens_to_ids(getattr(processor, "image_token", "<|image_pad|>"))
_tmpl = processor.chat_template if isinstance(processor.chat_template, str) else json.dumps(processor.chat_template)
TEMPLATE_KW = {"enable_thinking": False} if "enable_thinking" in (_tmpl or "") else {}
# HyperCLOVA X는 변수 이름이 thinking (기본값도 비사고 모드: 빈 <think></think> 블록). 명시적으로 끔
if not TEMPLATE_KW and _pname.startswith("HyperCLOVAX") and re.search(r"\bthinking\b", _tmpl or ""):
    TEMPLATE_KW = {"thinking": False}
SYSTEM_AS_STRING = False
_PROBE_ROW = {"question": "q", "a": "1", "b": "2", "c": "3", "d": "4"}
_raw_list = lambda t: "'type': 'text'" in t or '"type": "text"' in t      # 목록형 content가 풀리지 않고 글자 그대로 찍혔는지
try:
    _probe = processor.apply_chat_template(build_messages(_PROBE_ROW), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW); USE_SYSTEM_ROLE = True
    if _raw_list(_probe):                      # 템플릿이 목록형 system 내용을 그대로 찍음 (EXAONE 4.5 공식 템플릿 등)
        SYSTEM_AS_STRING = True
        _probe = processor.apply_chat_template(build_messages(_PROBE_ROW), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW)
        if _raw_list(_probe):
            SYSTEM_AS_STRING = False; USE_SYSTEM_ROLE = False; print("system 내용이 풀리지 않는 템플릿 → 지시문을 user 메시지 앞에 붙입니다")
        else:
            print("템플릿이 목록형 system 내용을 풀지 못함 → system 지시문을 문자열로 넣습니다")
except Exception as _e:
    USE_SYSTEM_ROLE = False; print("system 역할 미지원 템플릿 → 지시문을 user 메시지 앞에 붙입니다")
# 생각(추론) 모드가 열린 채 끝나는 템플릿(HyperCLOVA X 등)이면 빈 생각 블록을 닫아, 첫 토큰에서 바로 답하게 함
_probe = processor.apply_chat_template(build_messages(_PROBE_ROW), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW)
CLOSE_THINK = _probe.endswith("<think>\n") or _probe.endswith("<think>")
def _fix_prompt(t):
    if CLOSE_THINK and t.endswith("<think>\n"): return t + "\n</think>\n\n"
    if CLOSE_THINK and t.endswith("<think>"): return t + "\n\n</think>\n\n"
    return t
if CLOSE_THINK: print("⚠️ 템플릿이 생각 모드를 연 채 끝남 → 프롬프트 끝에 빈 생각 블록을 닫아 붙입니다")
print("프롬프트 미리보기(앞):", repr(_fix_prompt(_probe)[:220]))
print("프롬프트 미리보기(끝):", repr(_fix_prompt(_probe)[-90:]))
if _TRC and not hasattr(processor, "image_token_id") and not hasattr(processor, "image_token"):
    for _t in ("<image>", "<|image|>", "<|image_pad|>", "<IMG_CONTEXT>", "<img>"):
        if _t in (globals().get("_probe") or "") and tok.convert_tokens_to_ids(_t) not in (None, tok.unk_token_id):
            IMAGE_TOKEN_ID = tok.convert_tokens_to_ids(_t); break
print(f"계열: {FAMILY} | 이미지 토큰 id: {IMAGE_TOKEN_ID} | chat template 추가 인자: {TEMPLATE_KW} | system 역할: {USE_SYSTEM_ROLE}")

## 7-2. 모델 로드 + LoRA 부착 (비전 LoRA · 이어학습 선택)

In [ ]:
t0 = time.time()
if _TRC: load_kw["trust_remote_code"] = True
def _load(cls, kw):
    try: return cls.from_pretrained(MODEL_ID, **kw)
    except ValueError as e:
        _m = str(e).lower()
        if _TRC and kw.get("attn_implementation") == "sdpa" and ("sdpa" in _m or "scaled_dot_product_attention" in _m or "attn_implementation" in _m):
            print("sdpa 어텐션 미지원 모델 → eager로 로드"); return cls.from_pretrained(MODEL_ID, **{**kw, "attn_implementation": "eager"})
        raise
try:
    base = _load(AutoModelForImageTextToText, load_kw)
except (ValueError, KeyError) as e:
    if not _TRC: raise
    print("이미지-텍스트 자동 클래스로는 불러올 수 없는 커스텀 모델 → AutoModelForCausalLM으로 로드")
    base = _load(AutoModelForCausalLM, load_kw)
base.config.use_cache = False
if use_4bit and not BF16_NATIVE:
    try: base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
    except ValueError as e:
        if not _TRC: raise
        print("⚠️ 그래디언트 체크포인팅 미지원 모델 → 체크포인팅 없이 4bit 준비"); base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=False)
else:
    try: base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except Exception as e:
        if not _TRC: raise
        print("⚠️ 그래디언트 체크포인팅 미지원 모델 → 끄고 진행 (메모리 부족 시 마이크로 배치가 자동으로 줄어듭니다):", str(e)[:100])
print(f"모델 로드 {time.time() - t0:.0f}s | {type(base).__name__}")

LORA_TARGETS = r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj|in_proj_qkv|in_proj_z|out_proj)"
if _TRC and not any(re.fullmatch(LORA_TARGETS, n) for n, _ in base.named_modules()):
    LORA_TARGETS = r"(?i)^(?!.*(vision|visual|siglip|image|img|projector|connector|resampler|merger|abstractor|aligner)).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)"
    print("언어 모델 경로 이름이 달라('language_model' 없음) 비전·투영 모듈을 뺀 선형층에 LoRA를 붙입니다")
# 비전 쪽 모듈 판별: 'proj' = merger·투영층, 'tower' = 비전 인코더 블록, None = 언어 모델 등
_VIS_TOWER = re.compile(r"(^|\.)(visual|vision_tower|vision_model|vision_encoder)(\.|$)")
_VIS_PROJ = re.compile(r"(^|\.)(merger|multi_modal_projector|mm_projector|embed_vision|projector)(\.|$)")
def _vis_kind(name):
    if "language_model" in name or "lm_head" in name: return None
    if _VIS_PROJ.search(name): return "proj"
    if _VIS_TOWER.search(name): return "tower"
    return None
VIS_TARGETS = {"tower": [], "proj": []}
if VISION_LORA:
    _mods = dict(base.named_modules())
    for _n, _m in _mods.items():
        _k = _vis_kind(_n)
        if _k and isinstance(_m, torch.nn.Linear) and "patch_embed" not in _n:
            VIS_TARGETS[_k].append(_n)
    assert VIS_TARGETS["tower"], "비전 블록의 선형층을 찾지 못했습니다 → 이 모델의 비전 모듈 이름을 확인하세요 (_VIS_TOWER)"
    _vis_types = sorted({type(_mods[_n]).__name__ for _n in VIS_TARGETS["tower"] + VIS_TARGETS["proj"]})
    print(f"비전 LoRA 대상: 비전 블록 {len(VIS_TARGETS['tower'])}개 + merger/투영 {len(VIS_TARGETS['proj'])}개 | 층 종류 {_vis_types}")
    print("  예:", VIS_TARGETS["tower"][:4], VIS_TARGETS["proj"][:2])
    assert not any("4bit" in t or "8bit" in t for t in _vis_types), "비전 선형층이 양자화돼 있습니다(bf16이어야 함) → llm_int8_skip_modules를 확인하세요"
    from transformers.modeling_layers import GradientCheckpointingLayer
    _vblk = next((_m for _n, _m in _mods.items() if _vis_kind(_n) == "tower" and isinstance(_m, GradientCheckpointingLayer)), None)
    print("비전 블록 그래디언트 체크포인팅:", "켜짐" if (_vblk is not None and _vblk.gradient_checkpointing) else "꺼짐/해당 없음")
    _lm_names = [_n for _n in _mods if re.fullmatch(LORA_TARGETS, _n)]
    assert _lm_names, "언어 모델 LoRA 대상을 찾지 못했습니다"
    LORA_TARGETS = _lm_names + VIS_TARGETS["tower"] + VIS_TARGETS["proj"]      # 정확한 모듈 이름 목록 (PEFT가 이름 그대로 매칭)
    del _mods
RESUMING = os.path.exists(os.path.join(CKPT_DIR, "trainer_state.pt"))
INIT_ADAPTER_SHA = None
def _file_sha(p):
    _h = hashlib.sha256()
    with open(p, "rb") as f:
        for _b in iter(lambda: f.read(1 << 24), b""): _h.update(_b)
    return _h.hexdigest()[:16]
if RESUMING:
    model = PeftModel.from_pretrained(base, os.path.join(CKPT_DIR, "adapter"), is_trainable=True)
    print("↩️ 체크포인트에서 어댑터를 불러왔습니다:", CKPT_DIR)
elif INIT_ADAPTER:                                                       # 기존 어댑터에서 이어학습 (옵티마이저·스케줄러는 9번 셀에서 새로 만듦)
    assert os.path.isfile(os.path.join(INIT_ADAPTER, "adapter_model.safetensors")), f"시작 어댑터가 없습니다: {INIT_ADAPTER}"
    model = PeftModel.from_pretrained(base, INIT_ADAPTER, is_trainable=True)
    print("↪️ 기존 어댑터에서 이어학습:", INIT_ADAPTER)
else:
    torch.manual_seed(TRAIN_SEED)                                        # LoRA 초기화 시드
    model = get_peft_model(base, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
                                            target_modules=LORA_TARGETS))
# PEFT는 체크포인팅이 켜져 있으면 '입력 임베딩 출력에 grad 요구' 훅을 거는데, transformers 5.x에서는 이 훅이 비전 타워에도 걸려
# 얼려 둔 비전 타워 전체가 역전파 그래프에 들어갑니다(메모리·시간 낭비). 비재진입(non-reentrant) 체크포인팅에서는 필요 없으므로 제거합니다.
if hasattr(base, "disable_input_require_grads") and getattr(base, "_require_grads_hooks", None):
    base.disable_input_require_grads(); print("입력-grad 훅 제거 (비전 타워는 역전파에서 제외)")
if "qwen3_5" in str(getattr(base.config, "model_type", "")):
    import importlib.util
    if importlib.util.find_spec("fla") is None:
        print("⚠️⚠️ flash-linear-attention이 없습니다 → 선형 어텐션이 순수 PyTorch로 돌아 OOM/저속 위험. 2번 셀을 실행하고 '세션 다시 시작' 후 처음부터 실행하세요.")
    else:
        print("✅ flash-linear-attention 커널 사용")
if INIT_ADAPTER:
    INIT_ADAPTER_SHA = _file_sha(os.path.join(INIT_ADAPTER, "adapter_model.safetensors"))
    print("시작 어댑터 지문:", INIT_ADAPTER_SHA)
model.print_trainable_parameters()
n_lora = sum(1 for n, _ in model.named_modules() if n.endswith("lora_A"))
n_lora_vis = sum(1 for n, _ in model.named_modules() if n.endswith("lora_A") and (_vis_kind(n) or any(k in n for k in ("visual", "vision", "siglip"))))
print(f"LoRA가 붙은 층: {n_lora}개 (비전 쪽 {n_lora_vis}개 — " + ("비전 LoRA 켬)" if VISION_LORA else "0이어야 정상)"))
assert n_lora > 0 and ((n_lora_vis > 0) if VISION_LORA else (n_lora_vis == 0))
LORA_TARGET_COUNTS = {"language": n_lora - n_lora_vis, "vision": n_lora_vis}
_lc = ((_log.get("config") or {}).get("LORA_TARGET_COUNTS") if RECIPE and "_log" in globals() else None)
if _lc: print("LoRA 층 수가 학습 기록과 " + ("같음 ✅" if _lc == LORA_TARGET_COUNTS else f"다름 ⚠️ 기록 {_lc} · 지금 {LORA_TARGET_COUNTS}"))

LETTER_IDS = []                                     # 첫 번째(소문자·공백 없음)가 학습 목표, 나머지 변형은 확률 집계용
for L in LETTERS:
    ids = []
    for s in (L, L.upper(), " " + L, " " + L.upper()):
        t = tok.encode(s, add_special_tokens=False)
        if len(t) == 1 and t[0] not in ids: ids.append(t[0])
    assert ids and len(tok.encode(L, add_special_tokens=False)) == 1, f"'{L}' 가 단일 토큰이 아닙니다"
    LETTER_IDS.append(ids)
print("글자 토큰 id:", dict(zip(LETTERS, LETTER_IDS)))

## 8. 채점 함수 (답 자리 로짓 → 네 보기 확률) + 선지 순열 TTA

In [ ]:
IDENTITY = (0, 1, 2, 3)
TTA_PERMS = [(0, 1, 2, 3), (1, 2, 3, 0), (2, 3, 0, 1), (3, 0, 1, 2)]

def set_resolution(preset):
    global RES_PRESET, PIXEL_KW, GEN_SCALE
    RES_PRESET = preset
    GEN_SCALE = 1.0
    if FAMILY == "generic":
        PIXEL_KW = {}                                  # 커스텀 프로세서: 기본 전처리. up 프리셋은 원본 이미지를 확대해서 넘김
        GEN_SCALE = {"up1.5": 1.5, "up2.0": 2.0, "up2.5": 2.5}.get(preset, 1.0)
    elif FAMILY == "gemma4":
        PIXEL_KW = {"max_soft_tokens": int(RES_PRESETS_GEMMA4[preset])}
    elif FAMILY == "internvl":
        PIXEL_KW = {"crop_to_patches": True, "min_patches": 1, "max_patches": int(RES_PRESETS_INTERNVL[preset])}
    else:
        mn, mx = RES_PRESETS[preset]
        PIXEL_KW = {"min_pixels": int(mn), "max_pixels": int(mx)}
set_resolution(TRAIN_PRESET)

def render(row, perm=IDENTITY):
    return _fix_prompt(processor.apply_chat_template(build_messages(row, perm), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW))

from PIL import ImageEnhance
_AUG_ON = False                                   # train_micro_step 안에서만 True
AUG_RNG = np.random.RandomState(TRAIN_SEED + 12345)
def augment_image(im, rng=None):
    # 글자를 읽는 과제라 기하 변형(반전·회전·자르기)은 쓰지 않고, 촬영 조건에 해당하는 밝기·대비·채도·감마·JPEG 화질만 흔듦
    rng = rng or AUG_RNG
    im = ImageEnhance.Brightness(im).enhance(rng.uniform(0.8, 1.2))
    im = ImageEnhance.Contrast(im).enhance(rng.uniform(0.8, 1.2))
    im = ImageEnhance.Color(im).enhance(rng.uniform(0.9, 1.1))
    g = rng.uniform(0.85, 1.18); im = im.point(lambda v, g=g: int(255 * (v / 255) ** g))
    if rng.rand() < 0.5:
        import io; buf = io.BytesIO(); im.save(buf, "JPEG", quality=int(rng.uniform(60, 95))); buf.seek(0); im = Image.open(buf).convert("RGB")
    return im

def encode_batch(rows, perms):
    texts  = [render(r, p) for r, p in zip(rows, perms)]
    images = [load_image(r["path"]) for r in rows]
    if _AUG_ON and IMG_AUG: images = [augment_image(im) for im in images]
    if FAMILY == "generic" and GEN_SCALE != 1.0:
        images = [im.resize((round(im.width * GEN_SCALE), round(im.height * GEN_SCALE)), Image.BICUBIC) for im in images]
    if FAMILY == "gemma4": images = [[im] for im in images]          # Gemma 프로세서는 샘플별 이미지 리스트를 받음
    return processor(text=texts, images=images, padding=True, return_tensors="pt", **PIXEL_KW)

def _last_logits(enc):
    enc = enc.to(model.device)
    with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
        try:    out = model(**enc, logits_to_keep=1, use_cache=False)
        except TypeError: out = model(**enc)
    return out.logits[:, -1, :].float()

def unpermute(prob_slot, perms):
    probs = np.zeros_like(prob_slot)
    for i, p in enumerate(perms):
        for k, j in enumerate(p): probs[i, j] = prob_slot[i, k]
    return probs

@torch.inference_mode()
def score_rows(rows, perms=None):
    global BS
    perms = perms or [IDENTITY] * len(rows)
    enc = logits = None; oom = False
    try:
        enc = encode_batch(rows, perms)
        n_img_tok = (enc["input_ids"] == IMAGE_TOKEN_ID).sum(dim=1).tolist()
        logits = _last_logits(enc)
        if not torch.isfinite(logits).all(): raise FloatingPointError("로짓에 NaN/Inf 발생")
    except torch.cuda.OutOfMemoryError:
        oom = True
    if oom:
        del enc, logits; torch.cuda.empty_cache()
        if len(rows) == 1: raise RuntimeError("배치 1에서도 GPU 메모리 부족")
        h = len(rows) // 2; BS = max(1, min(BS, h)); print(f"  ⚠️ OOM → 평가 배치를 {BS}(으)로 줄입니다")
        p1, m1, t1 = score_rows(rows[:h], perms[:h]); p2, m2, t2 = score_rows(rows[h:], perms[h:])
        return np.concatenate([p1, p2]), np.concatenate([m1, m2]), t1 + t2
    logp = torch.log_softmax(logits, dim=-1)
    slot = torch.stack([torch.logsumexp(logp[:, ids], dim=1) for ids in LETTER_IDS], dim=1)
    mass = slot.exp().sum(dim=1).cpu().numpy()
    return unpermute(torch.softmax(slot, dim=1).cpu().numpy().astype(np.float64), perms), mass, n_img_tok

def score_rows_tta(rows, tta):
    ps, ms, nt = [], [], None
    for perm in (TTA_PERMS if tta else [IDENTITY]):
        p, m, nt = score_rows(rows, [perm] * len(rows)); ps.append(p); ms.append(m)
    return np.mean(ps, axis=0), np.mean(ms, axis=0), nt

def predict_df(df, tag, tta=False, resume=True, verbose=True, save_every=25):
    # df 전체를 채점. resume=True면 CSV에 이어 쓰며, 같은 tag로 다시 부르면 남은 문항만 처리
    was_training = model.training; model.eval()
    out_path = os.path.join(RUN_DIR, f"preds_{tag}__{RES_PRESET}{'__tta4' if tta else ''}__{len(df)}.csv")
    done_ids = set(pd.read_csv(out_path).id) if (resume and os.path.exists(out_path)) else set()
    if not resume and os.path.exists(out_path): os.remove(out_path)
    todo = df[~df.id.isin(done_ids)]
    if SKIP_MISSING_IMAGES: todo = todo[todo.path.map(lambda p: os.path.exists(img_path(p)))]
    rows = todo.to_dict("records"); recs, t0, n_done, nb = [], time.time(), 0, 0
    if verbose: print(f"[{os.path.basename(out_path)}] 전체 {len(df):,} | 완료 {len(done_ids):,} | 남음 {len(rows):,}")
    while n_done < len(rows):
        batch = rows[n_done: n_done + BS]
        probs, mass, ntok = score_rows_tta(batch, tta)
        for r, p, m, t in zip(batch, probs, mass, ntok):
            recs.append({"id": r["id"], "pred": LETTERS[int(p.argmax())], "p_a": p[0], "p_b": p[1], "p_c": p[2], "p_d": p[3],
                         "letter_mass": float(m), "img_tokens": int(t)})
        n_done += len(batch); nb += 1
        if nb % save_every == 0 or n_done >= len(rows):
            pd.DataFrame(recs).to_csv(out_path, mode="a", header=not os.path.exists(out_path), index=False); recs = []
            if verbose:
                el = time.time() - t0
                print(f"  {n_done:>5}/{len(rows)} | {el / n_done:.2f}s/문항 | 남은 시간 ~{(len(rows) - n_done) * el / n_done / 60:.0f}분", flush=True)
    if was_training: model.train()
    return pd.read_csv(out_path).drop_duplicates("id", keep="last"), out_path

CATS = {"전화번호": r"전화|번호|연락처", "가격/금액": r"가격|얼마|금액|요금|할인", "이름/상호/제목": r"이름|상호|제목|브랜드|가게|매장|간판|업체",
        "날짜/시간": r"날짜|언제|시간|기간|요일|몇 시|년도|연도", "위치/방향/층": r"어디|위치|방향|몇 층|층에|왼쪽|오른쪽|위쪽|아래",
        "개수/숫자": r"몇 개|몇 명|몇 번|개수|몇 가지|숫자", "색상": r"색|컬러", "부정형": r"아닌|않는|없는|틀린|잘못"}

def evaluate(df, tag, tta=False, resume=True, verbose=True, by_type=True):
    pred, _ = predict_df(df, tag, tta=tta, resume=resume, verbose=verbose)
    m = df.merge(pred, on="id", how="inner"); m["correct"] = (m.pred == m.answer)
    acc = float(m.correct.mean()); n = len(m); se = math.sqrt(max(acc * (1 - acc), 1e-9) / n)
    print(f"▶ {tag} | {RES_PRESET}{' +TTA4' if tta else ''} | 정확도 {acc * 100:.2f}% (n={n}, ±{1.96 * se * 100:.1f}%p) | letter_mass {m.letter_mass.mean():.3f}")
    if by_type:
        for k, pat in CATS.items():
            sel = m.question.str.contains(pat)
            if sel.sum() >= 10: print(f"    {k:<12} {m.correct[sel].mean() * 100:5.1f}%  (n={int(sel.sum())})")
    lp = os.path.join(OUT_DIR, "results_log_s2.csv")
    log = {"time": datetime.datetime.now().strftime("%m-%d %H:%M"), "run": f"{RUN_NAME}{'_dry' if DRY_RUN else ''}/{tag}", "model": str(MODEL_ID),
           "res": RES_PRESET, "tta": int(tta), "n": n, "acc": round(acc * 100, 2), "ci95": round(1.96 * se * 100, 2),
           "letter_mass": round(float(m.letter_mass.mean()), 3), "gpu": GPU_NAME, "split_md5": split_md5[:8]}
    pd.DataFrame([log]).to_csv(lp, mode="a", header=not os.path.exists(lp), index=False)
    return acc, m

## 9. 학습 준비: 배치 구성 · 손실 · 보조 학습 · 옵티마이저 · 체크포인트
옵티마이저는 언어 LoRA(`LR`)와 비전 LoRA(`LR x VIS_LR_SCALE`) 두 그룹입니다(비전 LoRA를 켰을 때).

In [ ]:
train_src = train_df if TRAIN_ON_FULL else tr_df
if SKIP_MISSING_IMAGES: train_src = train_src[train_src.path.map(lambda p: os.path.exists(img_path(p)))]
if DRY_RUN:       train_src = train_src.head(3 * MICRO_BS * GRAD_ACCUM)
elif TRAIN_LIMIT: train_src = train_src.head(int(TRAIN_LIMIT))
if EXTRA_TRAIN_CSV:                                   # 추가 학습 데이터(예: dev 의사 라벨)는 위 제한과 별개로 뒤에 붙임
    _need = ["id", "path", "question", "a", "b", "c", "d", "answer"]
    extra = pd.read_csv(EXTRA_TRAIN_CSV); assert all(c in extra.columns for c in _need), f"추가 데이터 열이 부족합니다: {list(extra.columns)}"
    for c in ["question", "a", "b", "c", "d"]: extra[c] = extra[c].astype(str)
    extra["answer"] = extra["answer"].astype(str).str.strip().str.lower(); assert set(extra.answer.unique()) <= set("abcd"), extra.answer.unique()
    assert not (set(extra.id) & set(val_df.id)), "추가 데이터에 검증 문항이 섞여 있습니다"
    assert extra.path.map(lambda p: os.path.exists(img_path(p))).all(), "추가 데이터의 이미지 경로가 없습니다"
    if DRY_RUN: extra = extra.head(2)
    train_src = pd.concat([train_src[_need], extra[_need]], ignore_index=True)
    print(f"추가 학습 데이터 {len(extra):,}문항 합침 (정답 분포 {extra.answer.value_counts().sort_index().to_dict()}) → 학습 총 {len(train_src):,}문항")
train_rows = train_src.to_dict("records")
STEP_SAMPLES = MICRO_BS * GRAD_ACCUM
steps_per_epoch = len(train_rows) // STEP_SAMPLES
n_epochs = 1 if DRY_RUN else EPOCHS
total_steps = steps_per_epoch * n_epochs
if MAX_STEPS and not DRY_RUN: total_steps = min(total_steps, int(MAX_STEPS))
assert total_steps > 0, "학습 샘플이 유효 배치보다 적습니다"
print(f"학습 문항 {len(train_rows):,} | 유효 배치 {STEP_SAMPLES} | epoch당 {steps_per_epoch} 스텝 | 총 {total_steps} 스텝" + (" | ⚠️ 검증셋 포함 학습(TRAIN_ON_FULL)" if TRAIN_ON_FULL else ""))

def epoch_plan(epoch):
    rng = np.random.RandomState(TRAIN_SEED * 1000 + epoch)
    order = rng.permutation(len(train_rows))
    perms = [tuple(rng.permutation(4).tolist()) if SHUFFLE_OPTIONS else IDENTITY for _ in range(len(train_rows))]
    return order, perms

def gold_slot(row, perm):                 # 섞인 화면에서 정답이 놓인 자리 (0~3)
    return perm.index(LETTERS.index(row["answer"]))

def train_micro_step(rows, perms):
    global _AUG_ON
    _AUG_ON = True
    try: enc = encode_batch(rows, perms)
    finally: _AUG_ON = False
    target = torch.tensor([LETTER_IDS[gold_slot(r, p)][0] for r, p in zip(rows, perms)], device=model.device)
    logits = _last_logits(enc)
    loss = F.cross_entropy(logits, target)
    with torch.no_grad():
        slot = torch.stack([logits[:, ids[0]] for ids in LETTER_IDS], dim=1)
        hit = (slot.argmax(1).cpu() == torch.tensor([gold_slot(r, p) for r, p in zip(rows, perms)])).float().mean().item()
    return loss, hit

# ── 문자열 보조학습: 이미지+질문을 보고 '정답 문자열'을 그대로 쓰는 과제. 정답 부분 토큰에만 손실, 문항마다 평균(긴 답이 지배하지 않게) ──
GEN_INSTRUCT = ("You are a helpful visual question answering assistant. "
                "Read the text in the image and answer with the exact words only. No explanation.")
if GEN_PROMPT_VERSION == "v2_ko_sys":
    GEN_INSTRUCT = ("당신은 이미지 속 글자를 읽고 질문에 답하는 도우미입니다. "
                    "선택지는 주어지지 않습니다. 이미지에 적힌 표현 그대로 짧게 답하고, 설명하지 마세요.")
aux_rows = []
if AUX_GEN_CSV:
    _aux = pd.read_csv(AUX_GEN_CSV)
    assert all(c in _aux.columns for c in ["id", "path", "question", "answer_text"]), f"보조 데이터 열이 부족합니다: {list(_aux.columns)}"
    _aux["question"] = _aux["question"].astype(str); _aux["answer_text"] = _aux["answer_text"].astype(str).str.strip()
    assert not (set(_aux.id) & set(val_df.id)), "보조 데이터에 검증 문항이 섞여 있습니다"
    assert _aux.path.map(lambda p: os.path.exists(img_path(p))).all(), "보조 데이터의 이미지 경로가 없습니다"
    _ntok = _aux.answer_text.map(lambda a: len(processor.tokenizer(a, add_special_tokens=False).input_ids))
    _aux = _aux[(_ntok >= 1) & (_ntok <= AUX_MAX_ANS_TOKENS) & (_aux.answer_text.str.len() > 0)]
    if DRY_RUN: _aux = _aux.head(4)
    aux_rows = _aux.to_dict("records")
    AUX_ORDER = np.random.RandomState(TRAIN_SEED + 777).permutation(len(aux_rows))
    print(f"문자열 보조학습 {len(aux_rows):,}문항 | 객관식 마이크로 배치 {AUX_GEN_EVERY}개마다 1개 | 가중치 {AUX_GEN_WEIGHT}")

def build_gen_messages(row):
    user_text = f"{row['question']}\n\n이미지에 적힌 그대로 짧게 답하세요."
    if globals().get("USE_SYSTEM_ROLE", True):
        _sys = GEN_INSTRUCT if globals().get("SYSTEM_AS_STRING", False) else [{"type": "text", "text": GEN_INSTRUCT}]
        return [{"role": "system", "content": _sys},
                {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]}]
    return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": GEN_INSTRUCT + "\n\n" + user_text}]}]

def gen_micro_step(rows):
    global _AUG_ON
    prompts = [_fix_prompt(processor.apply_chat_template(build_gen_messages(r), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW)) for r in rows]
    fulls = [p + r["answer_text"] for p, r in zip(prompts, rows)]
    _AUG_ON = True
    try: images = [load_image(r["path"]) for r in rows]; images = [augment_image(im) for im in images] if IMG_AUG else images
    finally: _AUG_ON = False
    imgs = [[im] for im in images] if FAMILY == "gemma4" else images
    enc_p = processor(text=prompts, images=imgs, padding=True, return_tensors="pt", **PIXEL_KW)
    enc = processor(text=fulls, images=imgs, padding=True, return_tensors="pt", **PIXEL_KW)
    n_ans = (enc["attention_mask"].sum(1) - enc_p["attention_mask"].sum(1)).tolist()     # 왼쪽 패딩이라 정답 토큰은 끝에 붙어 있음
    K = max(max(n_ans), 1) + 1
    enc = enc.to(model.device)
    with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
        out = model(**enc, logits_to_keep=K, use_cache=False)
    logits = out.logits.float(); ids = enc["input_ids"]; L = ids.shape[1]
    losses = []
    for b, n in enumerate(n_ans):
        if n < 1: continue
        tgt = ids[b, L - n:]
        losses.append(F.cross_entropy(logits[b, K - n - 1: K - 1], tgt))
    return torch.stack(losses).mean() if losses else None

def aux_batch(step, s):
    # 스텝·마이크로 위치로 결정적으로 고름 → 중단 후 재개해도 같은 보조 문항
    micro = step * max(1, STEP_SAMPLES // MICRO_BS) + s // MICRO_BS
    if not aux_rows or (micro + 1) % AUX_GEN_EVERY: return None
    g = (micro + 1) // AUX_GEN_EVERY - 1
    return [aux_rows[AUX_ORDER[(g * MICRO_BS + j) % len(aux_rows)]] for j in range(MICRO_BS)]

_named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
VIS_PARAMS = [p for n, p in _named if _vis_kind(n)]
LM_PARAMS = [p for n, p in _named if not _vis_kind(n)]
params = LM_PARAMS + VIS_PARAMS
assert bool(VIS_PARAMS) == bool(VISION_LORA), "비전 LoRA 설정과 학습 파라미터가 맞지 않습니다"
_groups = [{"params": LM_PARAMS, "lr": LR}] + ([{"params": VIS_PARAMS, "lr": LR * VIS_LR_SCALE}] if VIS_PARAMS else [])
optimizer = torch.optim.AdamW(_groups, lr=LR, weight_decay=0.0)
print(f"학습 파라미터: 언어 {sum(p.numel() for p in LM_PARAMS):,} (LR {LR:g})" + (f" | 비전 {sum(p.numel() for p in VIS_PARAMS):,} (LR {LR * VIS_LR_SCALE:g})" if VIS_PARAMS else ""))
scheduler = get_cosine_schedule_with_warmup(optimizer, max(1, int(total_steps * WARMUP_RATIO)), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16_SCALER)
state = {"global_step": 0, "history": [], "epoch_acc": {}, "config": {k: globals()[k] for k in
         ["MODEL_ID", "TRAIN_PRESET", "EPOCHS", "MAX_STEPS", "LR", "STEP_SAMPLES", "LORA_R", "LORA_ALPHA", "SHUFFLE_OPTIONS", "TRAIN_ON_FULL", "SEED", "TRAIN_SEED", "EXTRA_TRAIN_CSV", "IMG_AUG", "TRAIN_PRESETS", "AUX_GEN_CSV", "AUX_GEN_EVERY", "AUX_GEN_WEIGHT",
          "VISION_LORA", "VIS_LR_SCALE", "INIT_ADAPTER", "INIT_ADAPTER_SHA", "PLAN_EPOCH_OFFSET", "GEN_PROMPT_VERSION",
          "PROMPT_VERSION", "OCR_TAG", "OCR_MAX_CHARS", "OCR_POS_TAGS", "LORA_DROPOUT", "LORA_TARGET_COUNTS", "PRECISION"]}}

def save_checkpoint():
    tmp = CKPT_DIR + "_tmp"; os.makedirs(tmp, exist_ok=True)
    model.save_pretrained(os.path.join(tmp, "adapter"))
    rng = {"py": random.getstate(), "np": np.random.get_state(), "torch": torch.get_rng_state(),
           "cuda": torch.cuda.get_rng_state_all() if DEVICE == "cuda" else None}
    torch.save({"optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                "state": state, "rng": rng}, os.path.join(tmp, "trainer_state.pt"))
    if os.path.exists(CKPT_DIR): shutil.rmtree(CKPT_DIR)
    os.replace(tmp, CKPT_DIR)                     # 저장 도중 끊겨도 이전 체크포인트가 깨지지 않도록 교체 방식으로

if RESUMING:
    ck = torch.load(os.path.join(CKPT_DIR, "trainer_state.pt"), map_location="cpu", weights_only=False)
    _defaults = {"TRAIN_SEED": 42, "MAX_STEPS": None, "EXTRA_TRAIN_CSV": None, "IMG_AUG": False, "TRAIN_PRESETS": None, "AUX_GEN_CSV": None, "AUX_GEN_EVERY": 5, "AUX_GEN_WEIGHT": 1.0,
                 "VISION_LORA": False, "VIS_LR_SCALE": 0.2, "INIT_ADAPTER": None, "INIT_ADAPTER_SHA": None, "PLAN_EPOCH_OFFSET": 0, "GEN_PROMPT_VERSION": "v1_en_sys",
                 "PROMPT_VERSION": "baseline_v1"}     # 예전 체크포인트에 없는 항목의 기본값 (기록도 기본값도 없는 항목은 비교하지 않음)
    same = all(ck["state"]["config"].get(k, _defaults.get(k, v)) == v for k, v in state["config"].items())
    assert same, f"체크포인트의 설정이 현재와 다릅니다. RUN_NAME을 바꾸거나 설정을 되돌리세요.\n저장됨: {ck['state']['config']}\n현재: {state['config']}"
    optimizer.load_state_dict(ck["optimizer"]); scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
    state = ck["state"]
    random.setstate(ck["rng"]["py"]); np.random.set_state(ck["rng"]["np"]); torch.set_rng_state(ck["rng"]["torch"])
    if DEVICE == "cuda" and ck["rng"]["cuda"] is not None: torch.cuda.set_rng_state_all(ck["rng"]["cuda"])
    print(f"↩️ {state['global_step']}/{total_steps} 스텝부터 이어서 학습합니다")
if REUSE_FINISHED:                                   # 학습하지 않음: 끝난 실행의 기록(epoch별 검증 정확도)으로 11번 셀이 어댑터를 고름
    _old = json.load(open(os.path.join(RUN_DIR, "train_log.json"), encoding="utf-8"))
    state["epoch_acc"] = {int(k): v for k, v in (_old.get("epoch_acc") or {}).items()}
    state["global_step"] = total_steps
    print(f"↩️ 학습 없이 끝난 실행을 씁니다 (기록된 epoch 정확도 {state['epoch_acc']})")

## 10. 학습 루프
이어학습(`INIT_ADAPTER`)이면 먼저 시작 어댑터가 원래 모델과 같은 답을 내는지(검증 앞 40문항, 97.5% 이상) 확인합니다.

In [ ]:
_STOP_AT = None      # (테스트용) 이 스텝에서 강제 중단
quick_df = val_df.head(QUICK_EVAL_N if not DRY_RUN else 3 * BS)
if SKIP_MISSING_IMAGES: quick_df = quick_df[quick_df.path.map(lambda p: os.path.exists(img_path(p)))]
if ZEROSHOT_CHECK_N and not RESUMING:                 # 새 모델 점검: LoRA가 아직 0이라 기본 모델 그대로의 정확도
    set_resolution(TEST_PRESET if TEST_PRESET != "auto" else EVAL_PRESETS[-1]); model.eval()
    _zs_acc, _ = evaluate(val_df.head(int(ZEROSHOT_CHECK_N)), f"zeroshot_{RES_PRESET}", resume=False, verbose=False, by_type=False)
    print(f"🔎 학습 전(제로샷) 정확도 {_zs_acc * 100:.2f}% — 검증 앞 {int(ZEROSHOT_CHECK_N)}문항, 해상도 {RES_PRESET} "
          "(참고: 같은 방식으로 Qwen3.5-27B 96.0%, Gemma4-31B 97.7%)")
    set_resolution(TRAIN_PRESET)
if INIT_ADAPTER and not RESUMING and not REUSE_FINISHED:   # 이어학습 시작점 재현 확인: 원 어댑터의 저장된 검증 예측과 같아야 정상
    _ref_p = os.path.join(os.path.dirname(INIT_ADAPTER.rstrip("/")), "preds_final_ep1__up2.0__1000.csv")
    if os.path.exists(_ref_p):
        _ref = pd.read_csv(_ref_p).drop_duplicates("id", keep="last").set_index("id")
        set_resolution("up2.0"); model.eval()
        _chk = val_df.head(16 if DRY_RUN else 40)
        _pc, _ = predict_df(_chk, "init_check", resume=False, verbose=False); _pc = _pc.set_index("id").loc[_chk.id]
        _agree = float((_pc.pred.values == _ref.loc[_chk.id, "pred"].values).mean())
        _dmax = float(np.abs(_pc[["p_a", "p_b", "p_c", "p_d"]].values - _ref.loc[_chk.id, ["p_a", "p_b", "p_c", "p_d"]].values).max())
        print(f"🔎 이어학습 시작점 재현: 답 일치 {_agree * 100:.1f}% | 확률 최대 차 {_dmax:.4f} ({len(_chk)}문항, up2.0)")
        assert _agree >= 0.975, "시작 어댑터가 원 모델과 다르게 동작합니다 → INIT_ADAPTER 경로와 모델·정밀도 설정을 확인하세요"
        set_resolution(TRAIN_PRESET)
    else:
        print("⚠️ 시작 어댑터의 검증 예측 파일이 없어 재현 확인을 건너뜁니다:", _ref_p)
if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats()
model.train(); optimizer.zero_grad(set_to_none=True)
t_start, run_loss, run_hit, n_micro = time.time(), 0.0, 0.0, 0
run_aux, n_aux = 0.0, 0
step0 = state["global_step"]; plan_epoch = -1

for step in range(step0, total_steps):
    epoch, k = divmod(step, steps_per_epoch)
    if epoch != plan_epoch: order, perms_all = epoch_plan(epoch + PLAN_EPOCH_OFFSET); plan_epoch = epoch
    idx = order[k * STEP_SAMPLES: (k + 1) * STEP_SAMPLES]
    while True:                                        # OOM이면 마이크로 배치를 절반으로 줄여 같은 스텝을 다시 시도 (유효 배치는 그대로)
        oom = False; s_loss, s_hit, s_n, s_aux, s_aux_n = 0.0, 0.0, 0, 0.0, 0
        try:
            for s in range(0, STEP_SAMPLES, MICRO_BS):
                mb = idx[s: s + MICRO_BS]
                if TRAIN_PRESETS: set_resolution(TRAIN_PRESETS[np.random.RandomState(TRAIN_SEED * 100003 + step * 97 + s).randint(len(TRAIN_PRESETS))])   # 결정적 → 재개해도 같음
                loss, hit = train_micro_step([train_rows[i] for i in mb], [perms_all[i] for i in mb])
                scaler.scale(loss / GRAD_ACCUM).backward()
                s_loss += loss.item(); s_hit += hit; s_n += 1
                _ab = aux_batch(step, s)
                if _ab:
                    aux_loss = gen_micro_step(_ab)
                    if aux_loss is not None:
                        scaler.scale(aux_loss * AUX_GEN_WEIGHT / GRAD_ACCUM).backward()
                        s_aux += aux_loss.item(); s_aux_n += 1
        except torch.cuda.OutOfMemoryError:
            oom = True
        if not oom: break
        loss = None; optimizer.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()
        if MICRO_BS == 1: raise RuntimeError("마이크로 배치 1에서도 GPU 메모리 부족 → TRAIN_PRESET을 낮추거나 4bit를 켜세요.")
        MICRO_BS //= 2; GRAD_ACCUM *= 2
        print(f"  ⚠️ OOM → MICRO_BS={MICRO_BS}, GRAD_ACCUM={GRAD_ACCUM} 로 바꿔 이 스텝을 다시 실행합니다", flush=True)
    run_loss += s_loss; run_hit += s_hit; n_micro += s_n; run_aux += s_aux; n_aux += s_aux_n
    if TRAIN_PRESETS: set_resolution(TRAIN_PRESET)
    scaler.unscale_(optimizer)
    if VISION_LORA and step == step0:
        _gB = [float(p.grad.norm()) for n, p in model.named_parameters() if p.requires_grad and p.grad is not None and _vis_kind(n) and "lora_B" in n]
        _gA = [float(p.grad.norm()) for n, p in model.named_parameters() if p.requires_grad and p.grad is not None and _vis_kind(n) and "lora_A" in n]
        print(f"🔎 첫 스텝 비전 LoRA 그래디언트: B 합 {sum(_gB):.3e} ({len(_gB)}개) | A 합 {sum(_gA):.3e} — B가 0보다 크면 정상 (처음부터 학습할 땐 첫 스텝 A가 0이어도 정상)")
        assert sum(_gB) > 0, "비전 LoRA에 그래디언트가 흐르지 않습니다"
    gnorm = torch.nn.utils.clip_grad_norm_(params, 1.0)
    scaler.step(optimizer); scaler.update(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
    state["global_step"] = step + 1

    if (step + 1) % LOG_EVERY == 0 or step + 1 == total_steps or DRY_RUN:
        el = time.time() - t_start; done = step + 1 - step0; _lrs = "/".join(f"{x:.1e}" for x in scheduler.get_last_lr())
        print(f"step {step + 1:>4}/{total_steps} | ep {epoch + 1} | loss {run_loss / n_micro:.4f} | train_acc {run_hit / n_micro * 100:5.1f}% | "
              f"lr {_lrs} | gnorm {float(gnorm):.2f} | {el / done:.1f}s/step | 남은 시간 ~{(total_steps - step - 1) * el / done / 60:.0f}분"
              + (f" | 문자열 보조 loss {run_aux / n_aux:.3f}" if n_aux else ""), flush=True)
        state["history"].append({"step": step + 1, "loss": run_loss / n_micro, "train_acc": run_hit / n_micro, **({"aux_loss": run_aux / n_aux} if n_aux else {})})
        run_loss, run_hit, n_micro = 0.0, 0.0, 0; run_aux, n_aux = 0.0, 0
    end_of_epoch = (step + 1) % steps_per_epoch == 0 or step + 1 == total_steps      # MAX_STEPS로 중간에 끝나는 경우도 'epoch 끝'으로 취급
    if ((step + 1) % EVAL_EVERY == 0 and not end_of_epoch) or (DRY_RUN and step + 1 == total_steps):
        acc, _ = evaluate(quick_df, f"quick_step{step + 1}", resume=False, verbose=False, by_type=False)
        state["history"].append({"step": step + 1, "quick_val_acc": acc})
    if end_of_epoch and not DRY_RUN:
        if EPOCH_END_EVAL:
            acc, _ = evaluate(val_df, f"ep{epoch + 1}", resume=False, verbose=False)
            state["epoch_acc"][epoch + 1] = acc
        model.save_pretrained(os.path.join(RUN_DIR, f"adapter_ep{epoch + 1}"))
    if (step + 1) % SAVE_EVERY == 0 or end_of_epoch or step + 1 == total_steps:
        save_checkpoint()
    if _STOP_AT and step + 1 == _STOP_AT: print("강제 중단(테스트)"); break

if VISION_LORA and not REUSE_FINISHED:
    _wB = sum(float(p.detach().float().norm()) for n, p in model.named_parameters() if _vis_kind(n) and "lora_B" in n)
    print(f"🔎 비전 LoRA B 가중치 크기 합 {_wB:.3e} (0이면 비전 쪽이 학습되지 않은 것)")
    assert _wB > 0, "비전 LoRA 가중치가 처음 그대로입니다"
if DEVICE == "cuda": print(f"최대 VRAM 사용: {torch.cuda.max_memory_allocated() / 2**30:.1f}GB / {VRAM_GB:.0f}GB")
if DRY_RUN and total_steps > step0:
    per_step = (time.time() - t_start) / (total_steps - step0)
    full_steps = (len(train_df if TRAIN_ON_FULL else tr_df) // STEP_SAMPLES) * EPOCHS
    print(f"\n✅ DRY RUN 통과. 본 학습 예상: {full_steps} 스텝 x {per_step:.1f}s ≈ {full_steps * per_step / 3600:.1f}시간 (+검증 평가·test 추론 시간은 별도). DRY_RUN=False로 바꿔 1번 셀부터 다시 실행하세요.")
if not REUSE_FINISHED: json.dump({"epoch_acc": state["epoch_acc"], "history": state["history"], "config": state["config"], "recipe": RECIPE},
          open(os.path.join(RUN_DIR, "train_log.json"), "w"), indent=1, default=str)

## 11. 최종 평가: 가장 좋은 epoch의 어댑터 → 해상도별 검증 점수 + 오답 리포트
`DRY_RUN`에서는 저장한 어댑터를 다시 불러와 보고, test 추론 경로도 몇 문항으로 점검합니다.

In [ ]:
import base64, io, html as _html
def export_error_report(m, path, max_items=150):
    wrong = m[~m.correct].head(max_items); parts = [f"<h2>{_html.escape(os.path.basename(path))} — 오답 {int((~m.correct).sum())}건 / {len(m)}문항</h2>"]
    for _, r in wrong.iterrows():
        im = load_image(r.path); im.thumbnail((560, 560)); buf = io.BytesIO(); im.save(buf, "JPEG", quality=72)
        opts = "".join(f"<li>({k}) {_html.escape(str(r[k]))} — p={r['p_' + k]:.2f}{' <b>← 정답</b>' if k == r.answer else ''}{' <b style=color:#c00>← 예측</b>' if k == r.pred else ''}</li>" for k in LETTERS)
        parts.append(f"<div style='display:flex;gap:16px;margin:18px 0;border-top:1px solid #ccc;padding-top:12px'><img src='data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}'>"
                     f"<div><b>{r.id}</b><p>{_html.escape(str(r.question))}</p><ul>{opts}</ul>"
                     + (f"<pre style='white-space:pre-wrap;font-size:12px;background:#f5f5f5;padding:6px'>{_html.escape(ocr_block(r))}</pre>" if USE_OCR else "")
                     + "</div></div>")
    open(path, "w", encoding="utf-8").write("<meta charset='utf-8'><body style='font-family:sans-serif;max-width:1100px;margin:auto'>" + "".join(parts))
    print("오답 리포트 저장:", path)

if not DRY_RUN and state["global_step"] < total_steps:
    raise RuntimeError(f"학습이 아직 끝나지 않았습니다 ({state['global_step']}/{total_steps} 스텝). 1번 셀부터 다시 실행하면 체크포인트에서 이어서 학습합니다.")
if not DRY_RUN:
    best_ep = max(state["epoch_acc"], key=lambda e: (state["epoch_acc"][e], e)) if state["epoch_acc"] else n_epochs   # 동률이면 더 오래 학습한 쪽 · epoch 끝 평가를 껐으면 마지막 어댑터
    print("epoch별 검증 정확도:", {e: round(a * 100, 2) for e, a in state["epoch_acc"].items()}, "→ 사용할 어댑터: epoch", best_ep)
    set_peft_model_state_dict(model, load_peft_weights(os.path.join(RUN_DIR, f"adapter_ep{best_ep}"), device="cpu"))
    FINAL_TAG = f"final_ep{best_ep}"
else:
    FINAL_TAG = "final_dry"
model.eval()
final_eval_df = val_df if not DRY_RUN else quick_df
summary = []
for preset in (EVAL_PRESETS if not DRY_RUN else list(dict.fromkeys([TRAIN_PRESET] + list(EVAL_PRESETS)))):
    set_resolution(preset)
    if not DRY_RUN and preset == TRAIN_PRESET:                    # epoch 끝 평가와 같은 모델·해상도 → 그 결과를 그대로 재사용
        _src = os.path.join(RUN_DIR, f"preds_ep{best_ep}__{preset}__{len(final_eval_df)}.csv")
        _dst = os.path.join(RUN_DIR, f"preds_{FINAL_TAG}__{preset}__{len(final_eval_df)}.csv")
        if os.path.exists(_src) and not os.path.exists(_dst): shutil.copy(_src, _dst)
    acc, m = evaluate(final_eval_df, FINAL_TAG, tta=False)
    summary.append({"해상도": preset, "TTA": False, "정확도(%)": round(acc * 100, 2)})
    export_error_report(m, os.path.join(RUN_DIR, f"error_report_{FINAL_TAG}_{preset}.html"))
BEST_PRESET = max(summary, key=lambda s: s["정확도(%)"])["해상도"]        # 동률이면 앞에 적은(=더 싼) 해상도
if EVAL_TTA:
    set_resolution(BEST_PRESET)
    acc, _ = evaluate(final_eval_df, FINAL_TAG, tta=True, by_type=False)
    summary.append({"해상도": BEST_PRESET, "TTA": True, "정확도(%)": round(acc * 100, 2)})
print("\n" + pd.DataFrame(summary).to_string(index=False))
print(f"→ 제출 권장 해상도: {BEST_PRESET}")
if TRAIN_ON_FULL: print("⚠️ TRAIN_ON_FULL=True: 검증셋을 학습에 썼으므로 위 점수는 의미가 없습니다 (제출용 모델).")
if DRY_RUN:
    _ad = os.path.join(CKPT_DIR, "adapter")
    assert os.path.isdir(_ad), "DRY_RUN 체크포인트 어댑터가 저장되지 않았습니다"
    _w = load_peft_weights(_ad, device="cpu"); _vk = [k for k in _w if _vis_kind(k)]
    _res = set_peft_model_state_dict(model, _w); _unexp = list(getattr(_res, "unexpected_keys", None) or [])
    print(f"🔎 저장된 어댑터 다시 불러오기: 키 {len(_w)}개 (비전 {len(_vk)}개) | 예상 밖 키 {len(_unexp)}개")
    assert not _unexp and (bool(_vk) == bool(VISION_LORA)), "어댑터 저장/재로드 점검 실패"
    set_resolution(TEST_PRESET if TEST_PRESET != "auto" else EVAL_PRESETS[-1])
    _pt, _ = predict_df(test_df.head(2 * BS), "dry_test", resume=False, verbose=False)
    print(f"🔎 test 추론 경로 점검({RES_PRESET}): {len(_pt)}문항 | letter_mass 평균 {_pt.letter_mass.mean():.3f} | 이미지 토큰 평균 {_pt.img_tokens.mean():.0f}")
    assert len(_pt) == min(2 * BS, len(test_df)) and _pt.letter_mass.mean() > 0.9, "test 추론 경로 이상 (letter_mass가 낮음 → 7-1번 셀의 프롬프트 끝을 확인하세요)"
    print("✅ DRY_RUN 점검 모두 통과")
if REUSE_FINISHED and REPRO_CHECK_N:                  # 저장된 어댑터로 새로 채점 → 저장된 검증 예측과 대조 (재현성)
    _pre = "up2.0" if "up2.0" in EVAL_PRESETS else EVAL_PRESETS[0]
    _sp = os.path.join(RUN_DIR, f"preds_{FINAL_TAG}__{_pre}__{len(val_df)}.csv")
    if os.path.exists(_sp):
        set_resolution(_pre); _chk = val_df.head(int(REPRO_CHECK_N))
        _new, _ = predict_df(_chk, "repro_check", resume=False, verbose=False)
        _old = pd.read_csv(_sp).drop_duplicates("id", keep="last").set_index("id").loc[_chk.id]; _new = _new.set_index("id").loc[_chk.id]
        _ag = float((_new.pred.values == _old.pred.values).mean()); _dm = float(np.abs(_new[["p_a", "p_b", "p_c", "p_d"]].values - _old[["p_a", "p_b", "p_c", "p_d"]].values).max())
        REPRO = {"n": len(_chk), "agree": _ag, "max_prob_diff": _dm}
        print(f"🔎 재현 대조 ({_pre}, 검증 앞 {len(_chk)}문항): 저장된 예측과 답 일치 {_ag * 100:.1f}% · 확률 최대 차 {_dm:.4f} "
              "(같은 GPU 종류면 거의 0, GPU·라이브러리가 다르면 작은 차이는 정상)")
    else:
        print("⚠️ 재현 대조할 저장된 검증 예측이 없습니다:", _sp)
set_resolution(TRAIN_PRESET)

## 12. test 추론 → 제출 파일 (이어하기 지원)

In [ ]:
if MAKE_SUBMISSION and not DRY_RUN:
    test_preset = BEST_PRESET if TEST_PRESET == "auto" else TEST_PRESET
    set_resolution(test_preset)
    pred, _ = predict_df(test_df, f"{FINAL_TAG}_test", tta=TEST_TTA)
    sub = sub_df[["id"]].merge(pred[["id", "pred"]], on="id", how="left").rename(columns={"pred": "answer"})
    assert len(sub) == len(sub_df) and list(sub.id) == list(sub_df.id), "id 순서/개수 불일치"
    assert sub.answer.notna().all(), f"예측 누락 {int(sub.answer.isna().sum())}건"
    assert set(sub.answer.unique()) <= set(LETTERS), sub.answer.unique()
    sp = os.path.join(RUN_DIR, f"submission_{RUN_NAME}_{FINAL_TAG}_{test_preset}{'_tta4' if TEST_TTA else ''}.csv"); sub.to_csv(sp, index=False)
    print("저장:", sp, "| 예측 분포:", sub.answer.value_counts().sort_index().to_dict())
    set_resolution(TRAIN_PRESET)

## 12-2. (선택) dev 의사 라벨 만들기 — `*_dev` 레시피용
dev 문항(정답 없음, 교육생 응답 5개)을 이 실행의 모델로 채점해, **모델 확신도 ≥ `MIN_CONF`**이고 **응답 다수결 표 수 ≥ `MIN_VOTES`**이며 **모델 답 = 다수결 답**인 문항만 학습용 CSV로 저장합니다.
`gemma4_31b_r16_up20` 레시피에서 `MAKE_DEV_PSEUDO = True`로 실행하면 `dev_pseudo__gemma4_31b_r16_up20__up2.0__c0.8_v3_agree.csv`가 생기고, `*_dev` 레시피가 이 파일을 `EXTRA_TRAIN_CSV`로 씁니다.

In [ ]:
MAKE_DEV_PSEUDO = False                      # True: dev 채점 → 의사 라벨 CSV
DEV_PRESET, MIN_CONF, MIN_VOTES, REQUIRE_AGREE = "up2.0", 0.8, 3, True
if MAKE_DEV_PSEUDO and not DRY_RUN:
    P = ["p_a", "p_b", "p_c", "p_d"]
    assert dev_df is not None and len(dev_df), "dev.csv가 없습니다"
    for c in ["question", "a", "b", "c", "d"]: dev_df[c] = dev_df[c].astype(str)
    resp_cols = [c for c in dev_df.columns if c.startswith("answer")]         # answer1 ~ answer5 (교육생 응답)
    assert not (set(dev_df.id) & set(val_df.id)) and not (set(dev_df.id) & set(train_df.id)), "dev와 train/검증 id가 겹칩니다"
    set_resolution(DEV_PRESET); model.eval()
    pred, _ = predict_df(dev_df, "dev")                                     # RUN_DIR/preds_dev__{DEV_PRESET}__{len(dev)}.csv (이어하기 지원)
    m = dev_df.merge(pred, on="id", how="inner")
    def vote(row):
        v = [str(row[c]).strip().lower() for c in resp_cols]; v = [x for x in v if x in LETTERS]
        if not v: return None, 0
        cnt = {L: v.count(L) for L in set(v)}; top = max(cnt.values()); winners = [L for L, k in cnt.items() if k == top]
        return (winners[0] if len(winners) == 1 else None), top            # 동점이면 다수결 없음
    vv = [vote(r) for _, r in m.iterrows()]
    m["plural"] = [w for w, _ in vv]; m["n_votes"] = [k for _, k in vv]
    m["conf"] = m[P].max(axis=1)
    agree = m.pred == m.plural
    print(f"dev {len(m):,}문항 | 다수결 있음 {int(m.plural.notna().sum()):,} | 모델 = 다수결 {int(agree.sum()):,} ({agree.mean() * 100:.1f}%)")
    print("응답 다수결 표수별 모델 일치율:", {int(k): f"{agree[m.n_votes == k].mean() * 100:.0f}% (n={int((m.n_votes == k).sum())})" for k in sorted(m.n_votes.unique()) if k > 0})
    print("모델 확신도 분포:", {f"≥{t}": int((m.conf >= t).sum()) for t in (0.5, 0.7, 0.8, 0.9, 0.95)})
    keep = (m.conf >= MIN_CONF) & (m.n_votes >= MIN_VOTES)
    if REQUIRE_AGREE: keep &= agree
    out = m.loc[keep, ["id", "path", "question", "a", "b", "c", "d"]].copy(); out["answer"] = m.loc[keep, "pred"].values
    op = os.path.join(OUT_DIR, f"dev_pseudo__{RUN_NAME}__{DEV_PRESET}__c{MIN_CONF}_v{MIN_VOTES}{'_agree' if REQUIRE_AGREE else ''}.csv")
    out.to_csv(op, index=False)
    print(f"\n채택 {len(out):,}문항 / {len(m):,} | 정답 분포 {out.answer.value_counts().sort_index().to_dict()}\n저장: {op}")
    set_resolution(TRAIN_PRESET)
else:
    print("dev 의사 라벨: 끔 (MAKE_DEV_PSEUDO = False)" if not MAKE_DEV_PSEUDO else "DRY_RUN → 건너뜀")

## 13. 결과 요약

In [ ]:
print("RUN:", RUN_NAME, "| 레시피:", RECIPE, "| epoch별 검증 정확도(학습 해상도):", {e: round(a * 100, 2) for e, a in state["epoch_acc"].items()})
print(f"모델 {MODEL_ID} | 정밀도 {PRECISION} | 학습 해상도 {TRAIN_PRESET} | 유효 배치 {STEP_SAMPLES} | LR {LR:g} | 비전 LoRA {VISION_LORA} | OCR {USE_OCR} | "
      f"보조 학습 {'끔' if not AUX_GEN_CSV else os.path.basename(AUX_GEN_CSV)} | 이어학습 {'없음' if not INIT_ADAPTER else INIT_ADAPTER} | 전체 데이터 {TRAIN_ON_FULL}")
if "summary" in globals(): print("최종 검증:", summary)
h = pd.DataFrame([x for x in state["history"] if "loss" in x])
if len(h): print("loss 추이(처음→끝):", [round(v, 4) for v in h.loss.iloc[:: max(1, len(h) // 8)].tolist()])
q = [(x["step"], round(x["quick_val_acc"] * 100, 1)) for x in state["history"] if "quick_val_acc" in x]
if q: print("학습 중 빠른 검증(step, 정확도%):", q)
lp = os.path.join(OUT_DIR, "results_log_s2.csv")
if os.path.exists(lp):
    L = pd.read_csv(lp); L = L[L.run.astype(str).str.startswith(RUN_NAME)]
    print(L[[c for c in ["time", "run", "res", "tta", "n", "acc", "ci95", "letter_mass"] if c in L.columns]].tail(12).to_string(index=False))